# YOLO — worker 1 / seed 2

This is one of THREE different notebook files. Run each file once in a separate Kaggle session. The model, optimizer, split, runtime sources and remote checkpoint IDs are unchanged.

**Before starting these replacement copies, stop every older YOLO training copy and wait for its final HF upload.** The user confirmed the original YOLO session was stopped and uploaded. Also stop any other older YOLO worker that was started while experimenting with worker counts. Leave NB02, NB03 and NB05 running if desired.

This copy has `NUM_WORKERS=3`, `WORKER_ID=1`, and one-time `TAKE_OVER=True` to clear a leftover ownership lock. Do not launch it alongside another live copy owning seed 2. After a successful start, set TAKE_OVER back to False in the notebook configuration for future runs; changing that variable does not affect the already running child process.

Attach the same Phase 2 dataset, enable Internet and HF_TOKEN, select GPU and Run All. With one seed assigned, only one GPU is used even in a dual-T4 session. Existing progress is restored from its verified HF checkpoint; completed jobs are skipped. Never delete checkpoints or change runtime sources to bypass the lock.


# YOLO26 Medium segmentation training

**GPU T4 x2 · Internet ON · HF_TOKEN · attach [Tire Dataset Prepared phase2](https://www.kaggle.com/datasets/shanmuk4622/tire-dataset-prepared-phase2).**

NB00 has already passed and its HF report was independently verified. Do not rerun it. This notebook is self-contained; no GitHub repository upload is required. Run cells top to bottom.

Source dataset: 570 images. Frozen training overlay: **386 train / 81 validation / 103 test**, keeping all mid tyres and video frames together in train. This avoids unknown-identity overlap; held-out scores describe old low/high tyres only, not unseen mid/video performance.

Each model/condition has three seeds, 60 epochs. The default combined study is 15 runs across NB02–NB05. Optional `combined,old_only` gives the original 30-run A/B study at a larger compute cost.


This notebook verifies the source input, constructs the frozen overlay, downloads only pinned general-pretrained weights, checks real-model resume, and then trains. One independent process owns each GPU; data loading is synchronous and stateless for reproducible mid-epoch resume. Existing tyre-trained weights are not used for initialization.

In [ ]:
import os, sys, subprocess, base64, signal
from pathlib import Path
WORK = Path('/kaggle/working/phase2_training')
WORK.mkdir(parents=True, exist_ok=True)
RUNTIME = WORK / 'runtime'
RUNTIME.mkdir(exist_ok=True)
from kaggle_secrets import UserSecretsClient
TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
if not TOKEN:
    raise RuntimeError('Enable the HF_TOKEN Kaggle secret; never paste it into a cell.')
(RUNTIME / 'phase2_training_data.py').write_bytes(base64.b64decode('IiIiVmVyc2lvbmVkIHRyYWluaW5nIG92ZXJsYXk7IHNvdXJjZS12MSBpcyBhbHdheXMgcmVhZC1vbmx5LiIiIgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgbWF0aApmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBQSUwgaW1wb3J0IEltYWdlLCBJbWFnZUVuaGFuY2UsIEltYWdlRmlsdGVyCgpSRUxFQVNFX1NIQSA9ICcwNmM2YzlkMDAzYWZmODA4YmFiZjdlYmI1Nzg2ZjQ0MWU5NTA2NGViMDAwZjVlNWVkNWIxNTVkMGVlOWU3YzZhJwpSRVZJU0lPTiA9ICdwaGFzZTItdHJhaW5pbmctcjEnCk1PREVMUyA9IFsnbW9iaWxlbmV0djQnLCAncmVzbmV0NTAnLCAnc2VnZm9ybWVyJywgJ3lvbG8yNm0nLCAnaHJuZXQnXQpTUEVDUyA9IHsKICdtb2JpbGVuZXR2NCc6IGRpY3QobW9kZWw9J21vYmlsZW5ldHY0X2NvbnZfbWVkaXVtLmU1MDBfcjI1Nl9pbjFrJywgcmVwbz0ndGltbS9tb2JpbGVuZXR2NF9jb252X21lZGl1bS5lNTAwX3IyNTZfaW4xaycsIHJldmlzaW9uPSdhZDY2ODk4YzA0NWMxYjUyMjNlYTNmMmMwODMwYjc0Y2YyZTc1YmFjJywgZmlsZT0nbW9kZWwuc2FmZXRlbnNvcnMnLCBodz1bMzg0LDM4NF0sIGJhdGNoPTgpLAogJ3Jlc25ldDUwJzogZGljdChtb2RlbD0ncmVzbmV0NTAuYTFfaW4xaycsIHJlcG89J3RpbW0vcmVzbmV0NTAuYTFfaW4xaycsIHJldmlzaW9uPSc3NjcyNjg2MDNjYTBjYjBiZmUzMjZmYTg3Mjc3ZjE5YzQxOTU2NmVmJywgZmlsZT0nbW9kZWwuc2FmZXRlbnNvcnMnLCBodz1bMzg0LDM4NF0sIGJhdGNoPTgpLAogJ3NlZ2Zvcm1lcic6IGRpY3QobW9kZWw9J252aWRpYS9taXQtYjAnLCByZXBvPSdudmlkaWEvbWl0LWIwJywgcmV2aXNpb249JzgwOTgzYTQxM2MzMGQzNmEzOWMyMDIwMzk3NGFlNzgwNzgzNWUyYjQnLCBmaWxlPSdweXRvcmNoX21vZGVsLmJpbicsIGh3PVs1MTIsMzg0XSwgYmF0Y2g9MiksCiAnaHJuZXQnOiBkaWN0KG1vZGVsPSdocm5ldF93MTgubXNfYXVnX2luMWsnLCByZXBvPSd0aW1tL2hybmV0X3cxOC5tc19hdWdfaW4xaycsIHJldmlzaW9uPSc3ZTJjNTU4Mzc2OWY1NDUxNGZkODdlM2JhOWRlNDA4ZTMzZWFiYTBmJywgZmlsZT0nbW9kZWwuc2FmZXRlbnNvcnMnLCBodz1bNTEyLDM4NF0sIGJhdGNoPTIpLAogJ3lvbG8yNm0nOiBkaWN0KG1vZGVsPSd5b2xvMjZtLXNlZy55YW1sJywgdXJsPSdodHRwczovL2dpdGh1Yi5jb20vdWx0cmFseXRpY3MvYXNzZXRzL3JlbGVhc2VzL2Rvd25sb2FkL3Y4LjQuMC95b2xvMjZtLXNlZy5wdCcsIHNoYTI1Nj0nMTZiNjM2ZjA0ZThmYjZhMzI1YjMzNzBmMjJkYzVlNTUzNWZmNDczZTM4NGY0ZDA0MWZkMjhkNzg4ZjZlZTlmNScsIGh3PVs1MTIsMzg0XSwgYmF0Y2g9MiksCn0KCmRlZiBjYW5vbmljYWwoeCk6IHJldHVybiBqc29uLmR1bXBzKHgsIHNvcnRfa2V5cz1UcnVlLCBzZXBhcmF0b3JzPSgnLCcsICc6JykpLmVuY29kZSgpCmRlZiBkaWdlc3QoeCk6IHJldHVybiBoYXNobGliLnNoYTI1NihjYW5vbmljYWwoeCkpLmhleGRpZ2VzdCgpCmRlZiBzaGEocGF0aCk6CiBoPWhhc2hsaWIuc2hhMjU2KCkKIHdpdGggb3BlbihwYXRoLCdyYicpIGFzIGY6CiAgZm9yIGIgaW4gaXRlcihsYW1iZGE6Zi5yZWFkKDEwMjQqMTAyNCksYicnKTpoLnVwZGF0ZShiKQogcmV0dXJuIGguaGV4ZGlnZXN0KCkKZGVmIHJlYWQocGF0aCk6IHJldHVybiBqc29uLmxvYWRzKFBhdGgocGF0aCkucmVhZF90ZXh0KGVuY29kaW5nPSd1dGYtOCcpKQpkZWYgd3JpdGUocGF0aCxkYXRhKToKIHA9UGF0aChwYXRoKTtwLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKIHRtcD1wLndpdGhfc3VmZml4KHAuc3VmZml4KycudG1wJyk7dG1wLndyaXRlX2J5dGVzKGNhbm9uaWNhbChkYXRhKSk7dG1wLnJlcGxhY2UocCkKCmRlZiBsb2NhdGUocm9vdD0nJyk6CiBpZiByb290OiBwYXRocz1bUGF0aChyb290KV0KIGVsc2U6IHBhdGhzPVtwLnBhcmVudCBmb3IgcCBpbiBQYXRoKCcva2FnZ2xlL2lucHV0Jykucmdsb2IoJ1ZFUlNJT04uanNvbicpIGlmIHJlYWQocCkuZ2V0KCdkYXRhc2V0X3RpdGxlJyk9PSdUaXJlIERhdGFzZXQgUHJlcGFyZWQgcGhhc2UyJ10KIGlmIGxlbihwYXRocykhPTE6cmFpc2UgVmFsdWVFcnJvcignQXR0YWNoIGV4YWN0bHkgb25lIGV4cGFuZGVkIHRpcmUtZGF0YXNldC1wcmVwYXJlZC1waGFzZTIgaW5wdXQsIG9yIHNldCBEQVRBX1JPT1QuJykKIHJvb3Q9cGF0aHNbMF0ucmVzb2x2ZSgpCiBpZiBzaGEocm9vdC8nU0hBMjU2U1VNUy50eHQnKSE9UkVMRUFTRV9TSEE6cmFpc2UgVmFsdWVFcnJvcignV3JvbmcgZGF0YXNldCByZWxlYXNlOyBkbyBub3QgYnlwYXNzIHRoZSBjaGVja3N1bS4nKQogcmV0dXJuIHJvb3QKCmRlZiBvdmVybGF5KHJvb3QpOgogcm93cz1yZWFkKFBhdGgocm9vdCkvJ21hbmlmZXN0cy9pbWFnZXMuanNvbicpCiAjIEFsbCBwb3NzaWJsZSBtYXRjaGVzIG9mIGV2ZXJ5IHVucmVzb2x2ZWQgdmlkZW8gYXJlIHJlc3RyaWN0ZWQgdG8gVFJBSU4uCiAjIEhvbGRvdXRzIGFyZSB3aG9sZSBrbm93biBsb3cvaGlnaCB0eXJlcywgc2VsZWN0ZWQgYnkgY291bnRzLCBuZXZlciBieSBtb2RlbCByZXN1bHRzLgogb2xkPVtyIGZvciByIGluIHJvd3MgaWYgclsnZG9tYWluJ109PSdvcmlnaW5hbF9waG90byddCiBjb3VudHM9Q291bnRlcihyWydwaHlzaWNhbF90eXJlX2lkJ10gZm9yIHIgaW4gb2xkKQogY2xhc3Nlcz17clsncGh5c2ljYWxfdHlyZV9pZCddOnJbJ2NsYXNzX2luZGV4J10gZm9yIHIgaW4gb2xkfQogZ3JvdXBzPWRpY3QodHJhaW49W10sdmFsaWRhdGlvbj1bXSx0ZXN0PVtdKQogZm9yIGNscyBpbiAoMCwyKToKICBjYW5kaWRhdGVzPXNvcnRlZCgoZyBmb3IgZyBpbiBjb3VudHMgaWYgY2xhc3Nlc1tnXT09Y2xzIGFuZCBjb3VudHNbZ10+PTIwKSxrZXk9bGFtYmRhIGc6KC1jb3VudHNbZ10sZykpCiAgYXNzZXJ0IGxlbihjYW5kaWRhdGVzKT49MwogIGdyb3Vwc1sndmFsaWRhdGlvbiddLmFwcGVuZChjYW5kaWRhdGVzWzJdKTtncm91cHNbJ3Rlc3QnXS5hcHBlbmQoY2FuZGlkYXRlc1sxXSkKIGdyb3Vwc1sndHJhaW4nXT1zb3J0ZWQoc2V0KGNvdW50cyktc2V0KGdyb3Vwc1sndmFsaWRhdGlvbiddKS1zZXQoZ3JvdXBzWyd0ZXN0J10pKQogYXNzZXJ0IGFsbChnIGluIGdyb3Vwc1sndHJhaW4nXSBmb3IgZyBpbiBjb3VudHMgaWYgY2xhc3Nlc1tnXT09MSkKIGh1bWFuPXtyWydpbWFnZV9pZCddOnIgZm9yIHIgaW4gcmVhZChQYXRoKHJvb3QpLydnZW9tZXRyeS9vcmlnaW5hbF9odW1hbl9wb2ludHMuanNvbicpfQogcHJvcG9zZWQ9e3JbJ2ltYWdlX2lkJ106ciBmb3IgciBpbiByZWFkKFBhdGgocm9vdCkvJ2dlb21ldHJ5L25ld19wb2ludF9wcm9wb3NhbHMuanNvbicpfQogZm9yIHIgaW4gcm93czoKICByWydyb2xlJ109J3RyYWluJyBpZiByWydkb21haW4nXT09J25ld192aWRlb19mcmFtZScgZWxzZSBuZXh0KGsgZm9yIGssdiBpbiBncm91cHMuaXRlbXMoKSBpZiByWydwaHlzaWNhbF90eXJlX2lkJ10gaW4gdikKICBwPWh1bWFuLmdldChyWydpbWFnZV9pZCddKSBvciBwcm9wb3NlZC5nZXQoclsnaW1hZ2VfaWQnXSkKICByWydwb2ludF93ZWlnaHQnXT0xLiBpZiByWydpbWFnZV9pZCddIGluIGh1bWFuIGVsc2UgLjI1IGlmIHAgZWxzZSAwLgogIHJbJ3BvaW50cyddPVtxWyd4J10vKHJbJ3dpZHRoJ10tMSkgZm9yIHEgaW4gcFsncG9pbnRzJ11dIGlmIHAgZWxzZSBOb25lCiAgclsncG9pbnRfa2luZCddPSdodW1hbicgaWYgclsnaW1hZ2VfaWQnXSBpbiBodW1hbiBlbHNlICdwb2x5Z29uX3dlYWsnIGlmIHAgZWxzZSAnbm9uZScKICBpZiBwOmFzc2VydCBhbGwocVsneCddIGlzIG5vdCBOb25lIGZvciBxIGluIHBbJ3BvaW50cyddKQogYXNzZXJ0IGxlbihyb3dzKT09NTcwIGFuZCBsZW4oe3JbJ2ltYWdlX3NoYTI1NiddIGZvciByIGluIHJvd3N9KT09NTcwCiBhc3NlcnQgbm90IGFueShyWydkb21haW4nXT09J25ld192aWRlb19mcmFtZScgYW5kIHJbJ3JvbGUnXSE9J3RyYWluJyBmb3IgciBpbiByb3dzKQogcmV0dXJuIGRpY3QocmV2aXNpb249UkVWSVNJT04sc291cmNlX2NoZWNrc3VtPVJFTEVBU0VfU0hBLGdyb3Vwcz1ncm91cHMscm93cz1yb3dzLAogIGNvdW50cz1kaWN0KENvdW50ZXIoclsncm9sZSddIGZvciByIGluIHJvd3MpKSwKICBnZW9tZXRyeV9jb3VudHM9ZGljdChDb3VudGVyKHJbJ3JvbGUnXSBmb3IgciBpbiByb3dzIGlmIHJbJ3BvaW50cyddIGlzIG5vdCBOb25lKSksCiAgZGVjaXNpb25zPWRpY3QoaWRlbnRpdHk9J0FsbCB0aHJlZSBvcmlnaW5hbCBtaWQgdHlyZXMgYW5kIGFsbCBuZXcgZnJhbWVzIGFyZSB0cmFpbmluZy1vbmx5OyB1bmtub3duIGV4YWN0IG1hcHBpbmcgaXMgbm90IGludmVudGVkLicsCiAgIG1hc2tzPSdTZWdGb3JtZXIgaWdub3JlcyBjb250cmFkaWN0b3J5IHBpeGVscy4gWU9MTyB0cmFpbmluZyB0eXJlID0gcmF3IHR5cmUgT1IgcmF3IHRyZWFkOyB0cmVhZCB1bmNoYW5nZWQuIERlcml2ZWQgcG9saWN5LCBub3QgaHVtYW4gY29ycmVjdGlvbi4nLAogICBnZW9tZXRyeT0nTmV3IHBvbHlnb24tZGVyaXZlZCBwb2ludHMgYXJlIHdlYWsgdHJhaW5pbmcgdGFyZ2V0cyB3aXRoIHdlaWdodCAwLjI1OyB2YWxpZGF0aW9uL3Rlc3QgdXNlIG9ubHkgZXhpc3RpbmcgaHVtYW4gcG9pbnRzLicsCiAgIGV2YWx1YXRpb249J0hlbGQtb3V0IG9sZCBsb3cvaGlnaCB0eXJlcyBvbmx5LiBObyB1bnNlZW4tbWlkLCB1bnNlZW4tdmlkZW8sIHRocmVlLWNsYXNzLWdlbmVyYWxpc2F0aW9uIG9yIHNhZmV0eSBjbGFpbS4nKSkKCmRlZiB0cmFpbl9yb3dzKHBsYW4sam9iKToKIHJldHVybiBbciBmb3IgciBpbiBwbGFuWydyb3dzJ10gaWYgclsncm9sZSddPT0ndHJhaW4nIGFuZCAoam9iWydjb25kaXRpb24nXT09J2NvbWJpbmVkJyBvciByWydkb21haW4nXT09J29yaWdpbmFsX3Bob3RvJykgYW5kIChqb2JbJ21vZGVsJ10hPSdocm5ldCcgb3IgclsncG9pbnRzJ10gaXMgbm90IE5vbmUpXQoKZGVmIHNjaGVkdWxlKHBsYW4sam9iLGVwb2NoKToKICIiIlN0YXRlbGVzcyBjbGFzcy1iYWxhbmNlZCBzYW1wbGVyLCB0aGVuIGRvbWFpbi1iYWxhbmNlZCB3aXRoaW4gZWFjaCBjbGFzcy4iIiIKIHJvd3M9dHJhaW5fcm93cyhwbGFuLGpvYik7cnM9bnAucmFuZG9tLmRlZmF1bHRfcm5nKGpvYlsnc2VlZCddKjEwMDAwMDMrZXBvY2gpCiBwb29scz17fQogZm9yIGksciBpbiBlbnVtZXJhdGUocm93cyk6CiAgZ3JvdXA9clsncGh5c2ljYWxfdHlyZV9pZCddIG9yIHJbJ3NvdXJjZV92aWRlbyddCiAgcG9vbHMuc2V0ZGVmYXVsdChyWydjbGFzc19pbmRleCddLHt9KS5zZXRkZWZhdWx0KHJbJ2RvbWFpbiddLHt9KS5zZXRkZWZhdWx0KGdyb3VwLFtdKS5hcHBlbmQoaSkKICMgU2FtZSBvcHRpbWl6ZXItdXBkYXRlIGJ1ZGdldCBmb3Igb2xkLW9ubHkgYW5kIGNvbWJpbmVkIHdpdGhpbiBlYWNoIGFyY2hpdGVjdHVyZS4KIHJlZmVyZW5jZT10cmFpbl9yb3dzKHBsYW4sZGljdChqb2IsY29uZGl0aW9uPSdjb21iaW5lZCcpKQogYmF0Y2g9U1BFQ1Nbam9iWydtb2RlbCddXVsnYmF0Y2gnXTtzdGVwcz1tYXRoLmNlaWwobGVuKHJlZmVyZW5jZSkvYmF0Y2gpCiBvcmRlcj1bXQogZm9yIF8gaW4gcmFuZ2Uoc3RlcHMqYmF0Y2gpOgogIGNscz1pbnQocnMuY2hvaWNlKHNvcnRlZChwb29scykpKTtkb21haW5zPXBvb2xzW2Nsc107ZG9tYWluPXN0cihycy5jaG9pY2Uoc29ydGVkKGRvbWFpbnMpKSkKICBiYWdzPWRvbWFpbnNbZG9tYWluXTtncm91cD1zdHIocnMuY2hvaWNlKHNvcnRlZChiYWdzKSkpO29yZGVyLmFwcGVuZChpbnQocnMuY2hvaWNlKGJhZ3NbZ3JvdXBdKSkpCiByZXR1cm4gcm93cyxbb3JkZXJbaTppK2JhdGNoXSBmb3IgaSBpbiByYW5nZSgwLGxlbihvcmRlciksYmF0Y2gpXQoKZGVmIHNhbXBsZShyb290LHJvdyxodyxzZWVkPU5vbmUsbW9kZWw9J3NlZ2Zvcm1lcicpOgogIiIiU3luY2hyb25vdXMgZGV0ZXJtaW5pc3RpYyBDUFUgdHJhbnNmb3Jtczsgbm8gaGlkZGVuIHByZWZldGNoL1JORyBzdGF0ZS4iIiIKIHdpdGggSW1hZ2Uub3BlbihQYXRoKHJvb3QpL3Jvd1snaW1hZ2VfcGF0aCddKSBhcyBpbTppbT1pbS5jb252ZXJ0KCdSR0InKS5yZXNpemUodHVwbGUoaHdbOjotMV0pLEltYWdlLlJlc2FtcGxpbmcuQklMSU5FQVIpCiBtYXNrcz1bXQogZm9yIGtleSBpbiAoJ3R5cmVfbWFzaycsJ3RyZWFkX21hc2snLCdpZ25vcmVfbWFzaycpOgogIHdpdGggSW1hZ2Uub3BlbihQYXRoKHJvb3QpL3Jvd1trZXldKSBhcyBtOm1hc2tzLmFwcGVuZChucC5hc2FycmF5KG0ucmVzaXplKHR1cGxlKGh3Wzo6LTFdKSxJbWFnZS5SZXNhbXBsaW5nLk5FQVJFU1QpKT4wKQogbWFza3M9bnAuc3RhY2sobWFza3MpO3BvaW50cz1ucC5hcnJheShyb3dbJ3BvaW50cyddIG9yIFswLl0qNixkdHlwZT1ucC5mbG9hdDMyKQogaWYgc2VlZCBpcyBub3QgTm9uZToKICBycz1ucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZCkKICBpZiBycy5yYW5kb20oKTwuNToKICAgaW09aW0udHJhbnNwb3NlKEltYWdlLlRyYW5zcG9zZS5GTElQX0xFRlRfUklHSFQpO21hc2tzPW1hc2tzWzosOiw6Oi0xXS5jb3B5KCk7cG9pbnRzPTEtcG9pbnRzW1sxLDAsMywyLDUsNF1dCiAgaW09SW1hZ2VFbmhhbmNlLkJyaWdodG5lc3MoaW0pLmVuaGFuY2UoZmxvYXQocnMudW5pZm9ybSguOCwxLjIpKSkKICBpbT1JbWFnZUVuaGFuY2UuQ29udHJhc3QoaW0pLmVuaGFuY2UoZmxvYXQocnMudW5pZm9ybSguODUsMS4xNSkpKQogIGlmIHJzLnJhbmRvbSgpPC4yOmltPWltLmZpbHRlcihJbWFnZUZpbHRlci5HYXVzc2lhbkJsdXIoZmxvYXQocnMudW5pZm9ybSguMiwuOCkpKSkKIGlmIG1vZGVsPT0neW9sbzI2bSc6bWFza3NbMF18PW1hc2tzWzFdCiByZXR1cm4gbnAuYXNhcnJheShpbSxkdHlwZT1ucC5mbG9hdDMyKS50cmFuc3Bvc2UoMiwwLDEpLmNvcHkoKS8yNTUuLG1hc2tzLmNvcHkoKSxwb2ludHMKCmRlZiBqb2JzKGNvbmRpdGlvbnM9KCdjb21iaW5lZCcsKSxzZWVkcz0oMSwyLDMpKToKIHJldHVybiBbZGljdChtb2RlbD1tLGNvbmRpdGlvbj1jLHNlZWQ9cyxpZD1mJ3ttfS17Y30tc2VlZHtzfScpIGZvciBtIGluIE1PREVMUyBmb3IgYyBpbiBjb25kaXRpb25zIGZvciBzIGluIHNlZWRzXQo='))
(RUNTIME / 'phase2_models.py').write_bytes(base64.b64decode('IiIiRXhwbGljaXQgUGhhc2UgMiBtb2RlbCBhZGFwdGVycy4gTm8gb2xkIHR5cmUtdHJhaW5lZCBpbml0aWFsaXphdGlvbi4iIiIKZnJvbSBQSUwgaW1wb3J0IEltYWdlICAjIExvYWQgaW1hZ2luZyBETExzIGJlZm9yZSB0b3JjaCBvbiB0aGUgbG9jYWwgV2luZG93cyB0ZXN0IGhvc3QuCmltcG9ydCBjb3B5CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IG5uCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKZnJvbSBwaGFzZTJfdHJhaW5pbmdfZGF0YSBpbXBvcnQgU1BFQ1MsIHNhbXBsZQoKX0lOVEVSUE9MQVRFPUYuaW50ZXJwb2xhdGUKY2xhc3MgRGV0ZXJtaW5pc3RpY0JpbGluZWFyKHRvcmNoLmF1dG9ncmFkLkZ1bmN0aW9uKToKIEBzdGF0aWNtZXRob2QKIGRlZiBmb3J3YXJkKGN0eCx4LHNpemUpOgogIGN0eC5odz14LnNoYXBlWy0yOl07Y3R4LmR0eXBlPXguZHR5cGUKICByZXR1cm4gX0lOVEVSUE9MQVRFKHguZmxvYXQoKSxzaXplPXNpemUsbW9kZT0nYmlsaW5lYXInLGFsaWduX2Nvcm5lcnM9RmFsc2UpLnRvKHguZHR5cGUpCiBAc3RhdGljbWV0aG9kCiBkZWYgYmFja3dhcmQoY3R4LGcpOgogIGRlZiB3ZWlnaHRzKG4sbSk6CiAgIHA9KCh0b3JjaC5hcmFuZ2UobSxkZXZpY2U9Zy5kZXZpY2UsZHR5cGU9dG9yY2guZmxvYXQzMikrLjUpKm4vbS0uNSkuY2xhbXAoMCxuLTEpCiAgIGxvPXAuZmxvb3IoKS5sb25nKCk7aGk9KGxvKzEpLmNsYW1wX21heChuLTEpO2Y9cC1sbztjPXRvcmNoLmFyYW5nZShuLGRldmljZT1nLmRldmljZSkKICAgcmV0dXJuIChjW05vbmUsOl09PWxvWzosTm9uZV0pKigxLWZbOixOb25lXSkrKGNbTm9uZSw6XT09aGlbOixOb25lXSkqZls6LE5vbmVdCiAgd2l0aCB0b3JjaC5hdXRvY2FzdChnLmRldmljZS50eXBlLGVuYWJsZWQ9RmFsc2UpOgogICB5PXdlaWdodHMoY3R4Lmh3WzBdLGcuc2hhcGVbLTJdKTt4PXdlaWdodHMoY3R4Lmh3WzFdLGcuc2hhcGVbLTFdKTtyZXN1bHQ9eS5UQGcuZmxvYXQoKUB4CiAgcmV0dXJuIHJlc3VsdC50byhjdHguZHR5cGUpLE5vbmUKCmRlZiBkZXRlcm1pbmlzdGljX3Jlc2l6ZShpbnB1dCxzaXplPU5vbmUsc2NhbGVfZmFjdG9yPU5vbmUsbW9kZT0nbmVhcmVzdCcsYWxpZ25fY29ybmVycz1Ob25lLHJlY29tcHV0ZV9zY2FsZV9mYWN0b3I9Tm9uZSxhbnRpYWxpYXM9RmFsc2UpOgogaWYgbW9kZT09J2JpbGluZWFyJyBhbmQgaW5wdXQucmVxdWlyZXNfZ3JhZCBhbmQgdG9yY2guaXNfZ3JhZF9lbmFibGVkKCkgYW5kIGFsaWduX2Nvcm5lcnMgaXMgRmFsc2UgYW5kIHNpemUgaXMgbm90IE5vbmUgYW5kIHNjYWxlX2ZhY3RvciBpcyBOb25lIGFuZCBub3QgYW50aWFsaWFzOgogIHJldHVybiBEZXRlcm1pbmlzdGljQmlsaW5lYXIuYXBwbHkoaW5wdXQsKHNpemUsc2l6ZSkgaWYgaXNpbnN0YW5jZShzaXplLGludCkgZWxzZSB0dXBsZShzaXplKSkKIHJldHVybiBfSU5URVJQT0xBVEUoaW5wdXQsc2l6ZT1zaXplLHNjYWxlX2ZhY3Rvcj1zY2FsZV9mYWN0b3IsbW9kZT1tb2RlLGFsaWduX2Nvcm5lcnM9YWxpZ25fY29ybmVycyxyZWNvbXB1dGVfc2NhbGVfZmFjdG9yPXJlY29tcHV0ZV9zY2FsZV9mYWN0b3IsYW50aWFsaWFzPWFudGlhbGlhcykKCmRlZiBkZXRlcm1pbmlzdGljKCk6CiB0b3JjaC51c2VfZGV0ZXJtaW5pc3RpY19hbGdvcml0aG1zKFRydWUpCiB0b3JjaC5iYWNrZW5kcy5jdWRubi5iZW5jaG1hcms9RmFsc2U7dG9yY2guYmFja2VuZHMuY3Vkbm4uZGV0ZXJtaW5pc3RpYz1UcnVlCiB0b3JjaC5iYWNrZW5kcy5jdWRhLm1hdG11bC5hbGxvd190ZjMyPUZhbHNlO3RvcmNoLmJhY2tlbmRzLmN1ZG5uLmFsbG93X3RmMzI9RmFsc2UKIEYuaW50ZXJwb2xhdGU9ZGV0ZXJtaW5pc3RpY19yZXNpemUKCmNsYXNzIEdlb21ldHJ5KG5uLk1vZHVsZSk6CiBkZWYgX19pbml0X18oc2VsZik6CiAgc3VwZXIoKS5fX2luaXRfXygpO2ltcG9ydCB0aW1tCiAgc2VsZi5iYWNrYm9uZT10aW1tLmNyZWF0ZV9tb2RlbChTUEVDU1snaHJuZXQnXVsnbW9kZWwnXSxwcmV0cmFpbmVkPUZhbHNlLGZlYXR1cmVzX29ubHk9VHJ1ZSxmZWF0dXJlX2xvY2F0aW9uPScnLG91dF9pbmRpY2VzPSgxLDIsMyw0KSkKICBjaD1zZWxmLmJhY2tib25lLmZlYXR1cmVfaW5mby5jaGFubmVscygpO2Fzc2VydCBjaD09WzE4LDM2LDcyLDE0NF0KICBzZWxmLnByb2plY3Rpb25zPW5uLk1vZHVsZUxpc3Qobm4uQ29udjJkKGMsMTYsMSkgZm9yIGMgaW4gY2gpCiAgc2VsZi5oZWFkPW5uLlNlcXVlbnRpYWwobm4uQ29udjJkKDY0LDY0LDMscGFkZGluZz0xKSxubi5SZUxVKCksbm4uQ29udjJkKDY0LDYsMSkpCiBkZWYgZm9yd2FyZChzZWxmLHgpOgogIG1hcHM9c2VsZi5iYWNrYm9uZSh4KTtzaXplPW1hcHNbMF0uc2hhcGVbLTI6XQogIGhlYXQ9c2VsZi5oZWFkKHRvcmNoLmNhdChbRi5pbnRlcnBvbGF0ZShwKG0pLHNpemU9c2l6ZSxtb2RlPSduZWFyZXN0JykgZm9yIHAsbSBpbiB6aXAoc2VsZi5wcm9qZWN0aW9ucyxtYXBzKV0sMSkpCiAgbGluZXM9W10KICBmb3IgaSBpbiByYW5nZSg2KToKICAgcG9zPShoZWF0LnNoYXBlWy0yXS0xKSooWzM4NC8xNTM1LDc2OC8xNTM1LDExNTEvMTUzNV1baS8vMl0pO2xvPWludChwb3MpO2Y9cG9zLWxvCiAgIGxpbmVzLmFwcGVuZChoZWF0WzosaSxsbyw6XSooMS1mKStoZWF0WzosaSxtaW4obG8rMSxoZWF0LnNoYXBlWy0yXS0xKSw6XSpmKQogIHJldHVybiB0b3JjaC5zdGFjayhsaW5lcywxKQoKZGVmIGJ1aWxkKG5hbWUsYXNzZXRzLHByZXRyYWluZWQ9VHJ1ZSxkZXZpY2U9J2N1ZGEnKToKIHNwZWM9U1BFQ1NbbmFtZV07YXNzZXRzPVBhdGgoYXNzZXRzKQogaWYgbmFtZSBpbiAoJ21vYmlsZW5ldHY0JywncmVzbmV0NTAnLCdocm5ldCcpOgogIGltcG9ydCB0aW1tCiAgYXNzZXJ0IHRpbW0uX192ZXJzaW9uX189PScxLjAuMTUnCiAgbW9kZWw9R2VvbWV0cnkoKSBpZiBuYW1lPT0naHJuZXQnIGVsc2UgdGltbS5jcmVhdGVfbW9kZWwoc3BlY1snbW9kZWwnXSxwcmV0cmFpbmVkPUZhbHNlLG51bV9jbGFzc2VzPTMpCiAgaWYgcHJldHJhaW5lZDoKICAgZnJvbSBzYWZldGVuc29ycy50b3JjaCBpbXBvcnQgbG9hZF9maWxlCiAgIHN0YXRlPWxvYWRfZmlsZShzdHIoYXNzZXRzL25hbWUvJ21vZGVsLnNhZmV0ZW5zb3JzJykpCiAgIHRhcmdldD1tb2RlbC5iYWNrYm9uZSBpZiBuYW1lPT0naHJuZXQnIGVsc2UgbW9kZWwKICAga2V5cz10YXJnZXQuc3RhdGVfZGljdCgpOyBzZWxlY3RlZD17fQogICBmb3Igayx2IGluIGtleXMuaXRlbXMoKToKICAgIGlmIG5hbWUhPSdocm5ldCcgYW5kIGsuc3RhcnRzd2l0aCgoJ2NsYXNzaWZpZXIuJywnZmMuJykpOnNlbGVjdGVkW2tdPXYKICAgIGVsaWYgayBpbiBzdGF0ZSBhbmQgc3RhdGVba10uc2hhcGU9PXYuc2hhcGU6c2VsZWN0ZWRba109c3RhdGVba10KICAgIGVsaWYgay5lbmRzd2l0aCgnbnVtX2JhdGNoZXNfdHJhY2tlZCcpOnNlbGVjdGVkW2tdPXYKICAgIGVsc2U6cmFpc2UgVmFsdWVFcnJvcignUHJldHJhaW5lZCB0ZW5zb3IgbWlzbWF0Y2g6ICcraykKICAgdGFyZ2V0LmxvYWRfc3RhdGVfZGljdChzZWxlY3RlZCxzdHJpY3Q9VHJ1ZSkKIGVsaWYgbmFtZT09J3NlZ2Zvcm1lcic6CiAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IFNlZ2Zvcm1lckNvbmZpZyxTZWdmb3JtZXJGb3JTZW1hbnRpY1NlZ21lbnRhdGlvbixTZWdmb3JtZXJNb2RlbAogIGNmZz1TZWdmb3JtZXJDb25maWcuZnJvbV9wcmV0cmFpbmVkKHN0cihhc3NldHMvbmFtZSksbG9jYWxfZmlsZXNfb25seT1UcnVlKTtjZmcubnVtX2xhYmVscz0yCiAgbW9kZWw9U2VnZm9ybWVyRm9yU2VtYW50aWNTZWdtZW50YXRpb24oY2ZnKQogIGlmIHByZXRyYWluZWQ6CiAgIGVuY29kZXI9U2VnZm9ybWVyTW9kZWwuZnJvbV9wcmV0cmFpbmVkKHN0cihhc3NldHMvbmFtZSksbG9jYWxfZmlsZXNfb25seT1UcnVlLHVzZV9zYWZldGVuc29ycz1GYWxzZSkKICAgbW9kZWwuc2VnZm9ybWVyLmxvYWRfc3RhdGVfZGljdChlbmNvZGVyLnN0YXRlX2RpY3QoKSxzdHJpY3Q9VHJ1ZSk7ZGVsIGVuY29kZXIKIGVsaWYgbmFtZT09J3lvbG8yNm0nOgogIGZyb20gdWx0cmFseXRpY3Mubm4udGFza3MgaW1wb3J0IFNlZ21lbnRhdGlvbk1vZGVsCiAgZnJvbSB1bHRyYWx5dGljcy5jZmcgaW1wb3J0IGdldF9jZmcKICBtb2RlbD1TZWdtZW50YXRpb25Nb2RlbChzcGVjWydtb2RlbCddLGNoPTMsbmM9Mix2ZXJib3NlPUZhbHNlKQogIG1vZGVsLmFyZ3M9Z2V0X2NmZyhvdmVycmlkZXM9ZGljdCh0YXNrPSdzZWdtZW50JyxvdmVybGFwX21hc2s9RmFsc2UsZXBvY2hzPTYwLGJveD03LjUsY2xzPS41LGRmbD0xLjUpKQogIGlmIHByZXRyYWluZWQ6CiAgICMgT2ZmaWNpYWwgdXBzdHJlYW0gYXNzZXQgb25seTsgZmlsZSBoYXNoIGlzIGZyb3plbiBpbiB0aGUgcnVuIGNvbnRyYWN0LgogICBja3B0PXRvcmNoLmxvYWQoYXNzZXRzL25hbWUvJ3lvbG8yNm0tc2VnLnB0JyxtYXBfbG9jYXRpb249J2NwdScsd2VpZ2h0c19vbmx5PUZhbHNlKQogICBvcmlnaW5hbD1ja3B0LmdldCgnZW1hJykgb3IgY2twdFsnbW9kZWwnXTttb2RlbC5sb2FkKG9yaWdpbmFsLmZsb2F0KCksdmVyYm9zZT1GYWxzZSk7ZGVsIGNrcHQsb3JpZ2luYWwKICAjIE5hdGl2ZSBZT0xPMjYgYXV4aWxpYXJ5IHNlbWFudGljIGxvc3MgYXNzdW1lcyBleGNsdXNpdmUgY2xhc3Nlcy4gT3VyIG5lc3RlZAogICMgdHlyZS90cmVhZCBpbnN0YW5jZXMgcmVtYWluIG92ZXJsYXBwaW5nOyBkaXNhYmxlIG9ubHkgdGhhdCBhdXhpbGlhcnkgYnJhbmNoLgogICMgSW5zdGFuY2UgbWFza3MsIGJvdGggZGV0ZWN0aW9uIGhlYWRzIGFuZCB0aGVpciBvcmRpbmFyeSBsb3NzZXMgcmVtYWluIGFjdGl2ZS4KICBtb2RlbC5uYW1lcz17MDondHlyZScsMTondHJlYWQnfQogZWxzZTpyYWlzZSBWYWx1ZUVycm9yKG5hbWUpCiBtb2RlbD1tb2RlbC50byhkZXZpY2UpCiBpZiBuYW1lPT0neW9sbzI2bSc6bW9kZWwuY3JpdGVyaW9uPW1vZGVsLmluaXRfY3JpdGVyaW9uKCkKIHJldHVybiBtb2RlbAoKZGVmIHRyYWluaW5nX21vZGUobW9kZWwpOgogbW9kZWwudHJhaW4oKQogZm9yIGxheWVyIGluIG1vZGVsLm1vZHVsZXMoKToKICBpZiBpc2luc3RhbmNlKGxheWVyLG5uLm1vZHVsZXMuYmF0Y2hub3JtLl9CYXRjaE5vcm0pOmxheWVyLmV2YWwoKQoKZGVmIGJhdGNoKHJvb3Qscm93cyxuYW1lLGVwb2NoPTAsc3RlcD0wLHNlZWQ9Tm9uZSxkZXZpY2U9J2N1ZGEnKToKIGFycmF5cz1bc2FtcGxlKHJvb3QscixTUEVDU1tuYW1lXVsnaHcnXSxOb25lIGlmIHNlZWQgaXMgTm9uZSBlbHNlIHNlZWQqMTAwMDAwMTkrZXBvY2gqMTAwMDAzK3N0ZXAqOTcraSxuYW1lKSBmb3IgaSxyIGluIGVudW1lcmF0ZShyb3dzKV0KIHg9dG9yY2gudGVuc29yKG5wLnN0YWNrKFt2WzBdIGZvciB2IGluIGFycmF5c10pLGRldmljZT1kZXZpY2UpCiBtYXNrcz10b3JjaC50ZW5zb3IobnAuc3RhY2soW3ZbMV0gZm9yIHYgaW4gYXJyYXlzXSksZGV2aWNlPWRldmljZSkKIHBvaW50cz10b3JjaC50ZW5zb3IobnAuc3RhY2soW3ZbMl0gZm9yIHYgaW4gYXJyYXlzXSksZGV2aWNlPWRldmljZSkKIGlmIG5hbWUhPSd5b2xvMjZtJzp4PSh4LXgubmV3X3RlbnNvcihbLjQ4NSwuNDU2LC40MDZdKVtOb25lLDosTm9uZSxOb25lXSkveC5uZXdfdGVuc29yKFsuMjI5LC4yMjQsLjIyNV0pW05vbmUsOixOb25lLE5vbmVdCiByZXN1bHQ9ZGljdChpbWc9eCxtYXNrcz1tYXNrcyxwb2ludHM9cG9pbnRzLGxhYmVscz10b3JjaC50ZW5zb3IoW3JbJ2NsYXNzX2luZGV4J10gZm9yIHIgaW4gcm93c10sZGV2aWNlPWRldmljZSkscG9pbnRfd2VpZ2h0PXgubmV3X3RlbnNvcihbclsncG9pbnRfd2VpZ2h0J10gZm9yIHIgaW4gcm93c10pKQogaWYgbmFtZT09J3lvbG8yNm0nOgogIGJveGVzPVtdO2luZGljZXM9W107Y2xhc3Nlcz1bXTtpbnN0YW5jZXM9W107aCx3PXguc2hhcGVbLTI6XQogIGZvciBpIGluIHJhbmdlKGxlbihyb3dzKSk6CiAgIGZvciBjIGluICgwLDEpOgogICAgeSx6PXRvcmNoLndoZXJlKG1hc2tzW2ksY10pO2Fzc2VydCBsZW4oeiksJ0VtcHR5IHJlc2l6ZWQgWU9MTyB0YXJnZXQnCiAgICB4MCx4MT16Lm1pbigpLHoubWF4KCkrMTt5MCx5MT15Lm1pbigpLHkubWF4KCkrMQogICAgYm94ZXMuYXBwZW5kKHRvcmNoLnN0YWNrKCgoeDAreDEpLzIvdywoeTAreTEpLzIvaCwoeDEteDApL3csKHkxLXkwKS9oKSkpCiAgICBpbmRpY2VzLmFwcGVuZChpKTtjbGFzc2VzLmFwcGVuZChjKTtpbnN0YW5jZXMuYXBwZW5kKG1hc2tzW2ksY10uZmxvYXQoKSkKICByZXN1bHQudXBkYXRlKGJib3hlcz10b3JjaC5zdGFjayhib3hlcyksYmF0Y2hfaWR4PXRvcmNoLnRlbnNvcihpbmRpY2VzLGRldmljZT1kZXZpY2UpLGNscz14Lm5ld190ZW5zb3IoY2xhc3NlcykudmlldygtMSwxKSxtYXNrcz10b3JjaC5zdGFjayhpbnN0YW5jZXMpKQogcmV0dXJuIHJlc3VsdAoKZGVmIHdpdGhvdXRfc2VtYW50aWMocHJlZCk6CiBpZiBpc2luc3RhbmNlKHByZWQsZGljdCk6CiAgb3V0PXtrOndpdGhvdXRfc2VtYW50aWModikgZm9yIGssdiBpbiBwcmVkLml0ZW1zKCl9CiAgaWYgaXNpbnN0YW5jZShvdXQuZ2V0KCdwcm90bycpLHR1cGxlKTpvdXRbJ3Byb3RvJ109b3V0Wydwcm90byddWzBdCiAgcmV0dXJuIG91dAogcmV0dXJuIHByZWQKCmRlZiBsb3NzKG1vZGVsLGIsbmFtZSxlcG9jaCk6CiBpZiBuYW1lIGluICgnbW9iaWxlbmV0djQnLCdyZXNuZXQ1MCcpOnJldHVybiBGLmNyb3NzX2VudHJvcHkobW9kZWwoYlsnaW1nJ10pLmZsb2F0KCksYlsnbGFiZWxzJ10sbGFiZWxfc21vb3RoaW5nPS4wNSkKIGlmIG5hbWU9PSdocm5ldCc6CiAgej1tb2RlbChiWydpbWcnXSkuZmxvYXQoKTtncmlkPXRvcmNoLmFyYW5nZSh6LnNoYXBlWy0xXSxkZXZpY2U9ei5kZXZpY2UpCiAgdGFyZ2V0PXRvcmNoLmV4cCgtLjUqKChncmlkLWJbJ3BvaW50cyddWy4uLixOb25lXSooei5zaGFwZVstMV0tMSkpLzEuNSkqKjIpCiAgdGFyZ2V0PXRhcmdldC90YXJnZXQuc3VtKC0xLGtlZXBkaW09VHJ1ZSkuY2xhbXBfbWluKDFlLTEyKQogIHZhbHVlcz0tKHRhcmdldCp6LmxvZ19zb2Z0bWF4KC0xKSkuc3VtKC0xKS5tZWFuKC0xKQogIHJldHVybiAodmFsdWVzKmJbJ3BvaW50X3dlaWdodCddKS5tZWFuKCkKIGlmIG5hbWU9PSdzZWdmb3JtZXInOgogIHo9Ri5pbnRlcnBvbGF0ZShtb2RlbChwaXhlbF92YWx1ZXM9YlsnaW1nJ10pLmxvZ2l0cy5mbG9hdCgpLGJbJ21hc2tzJ10uc2hhcGVbLTI6XSxtb2RlPSdiaWxpbmVhcicsYWxpZ25fY29ybmVycz1GYWxzZSkKICB5PWJbJ21hc2tzJ11bOiw6Ml0uZmxvYXQoKTt2YWxpZD0ofmJbJ21hc2tzJ11bOiwyOjNdKS5mbG9hdCgpO3A9ei5zaWdtb2lkKCkKICBiY2U9KEYuYmluYXJ5X2Nyb3NzX2VudHJvcHlfd2l0aF9sb2dpdHMoeix5LHJlZHVjdGlvbj0nbm9uZScpKnZhbGlkKS5zdW0oKS8odmFsaWQuc3VtKCkqMikuY2xhbXBfbWluKDEpCiAgZGltcz0oMCwyLDMpO2RpY2U9KDIqKHAqeSp2YWxpZCkuc3VtKGRpbXMpKzEpLygocCp2YWxpZCkuc3VtKGRpbXMpKyh5KnZhbGlkKS5zdW0oZGltcykrMSkKICByZXR1cm4gYmNlKzEtZGljZS5tZWFuKCkKIGlmIG5hbWU9PSd5b2xvMjZtJzoKICBjcml0ZXJpb249bW9kZWwuY3JpdGVyaW9uCiAgIyBUaGUgdXBzdHJlYW0gcHJvZ3Jlc3NpdmUgbG9zcyBoYXMgbXV0YWJsZSBlcG9jaCB3ZWlnaHRpbmcuIFNldCBleHBsaWNpdGx5CiAgIyBmcm9tIHNhdmVkIGVwb2NoIHJhdGhlciB0aGFuIHJlbHlpbmcgb24gZnJhbWV3b3JrIGNhbGxiYWNrcy4KICBpZiBoYXNhdHRyKGNyaXRlcmlvbiwnbzJtJyk6CiAgIGNyaXRlcmlvbi5vMm09Y3JpdGVyaW9uLmRlY2F5KGVwb2NoKTtjcml0ZXJpb24ubzJvPTEtY3JpdGVyaW9uLm8ybQogIHByZWQ9d2l0aG91dF9zZW1hbnRpYyhtb2RlbChiWydpbWcnXSkpCiAgcmV0dXJuIGNyaXRlcmlvbihwcmVkLGIpWzBdLnN1bSgpL2xlbihiWydpbWcnXSkKIHJhaXNlIFZhbHVlRXJyb3IobmFtZSkKCmRlZiBwcmVkaWN0KG1vZGVsLGIsbmFtZSk6CiBpZiBuYW1lIGluICgnbW9iaWxlbmV0djQnLCdyZXNuZXQ1MCcpOnJldHVybiBtb2RlbChiWydpbWcnXSkuZmxvYXQoKS5zb2Z0bWF4KC0xKQogaWYgbmFtZT09J2hybmV0JzoKICB6PW1vZGVsKGJbJ2ltZyddKS5mbG9hdCgpO3JldHVybiAoei5zb2Z0bWF4KC0xKSp0b3JjaC5saW5zcGFjZSgwLDEsei5zaGFwZVstMV0sZGV2aWNlPXouZGV2aWNlKSkuc3VtKC0xKQogaWYgbmFtZT09J3NlZ2Zvcm1lcic6cmV0dXJuIEYuaW50ZXJwb2xhdGUobW9kZWwocGl4ZWxfdmFsdWVzPWJbJ2ltZyddKS5sb2dpdHMuZmxvYXQoKSxiWydpbWcnXS5zaGFwZVstMjpdLG1vZGU9J2JpbGluZWFyJyxhbGlnbl9jb3JuZXJzPUZhbHNlKS5zaWdtb2lkKCk+PS41CiBpZiBuYW1lPT0neW9sbzI2bSc6CiAgZnJvbSB1bHRyYWx5dGljcy51dGlscy5vcHMgaW1wb3J0IHByb2Nlc3NfbWFzawogIG91dD1tb2RlbChiWydpbWcnXSk7cGFpcj1vdXRbMF0gaWYgaXNpbnN0YW5jZShvdXRbMF0sdHVwbGUpIGVsc2Ugb3V0CiAgZGV0ZWN0aW9ucyxwcm90bz1wYWlyCiAgcmVzdWx0PXRvcmNoLnplcm9zKChsZW4oYlsnaW1nJ10pLDIsKmJbJ2ltZyddLnNoYXBlWy0yOl0pLGR0eXBlPXRvcmNoLmJvb2wsZGV2aWNlPWJbJ2ltZyddLmRldmljZSkKICBmb3IgaSxkZXQgaW4gZW51bWVyYXRlKGRldGVjdGlvbnMpOgogICBkZXQ9ZGV0W2RldFs6LDRdPj0uMjVdCiAgIGlmIG5vdCBsZW4oZGV0KTpjb250aW51ZQogICBtYXNrcz1wcm9jZXNzX21hc2socHJvdG9baV0sZGV0WzosNjpdLGRldFs6LDo0XSxiWydpbWcnXS5zaGFwZVstMjpdLHVwc2FtcGxlPVRydWUpLmJvb2woKQogICBmb3IgYyBpbiAoMCwxKToKICAgIGhpdD1kZXRbOiw1XS5sb25nKCk9PWMKICAgIGlmIGhpdC5hbnkoKTpyZXN1bHRbaSxjXT1tYXNrc1toaXRdLmFueSgwKQogIHJldHVybiByZXN1bHQK'))
(RUNTIME / 'phase2_worker.py').write_bytes(base64.b64decode('IiIiT25lIEdQVSBwcm9jZXNzLCBkdXJhYmxlIG9wdGltaXplci1zdGVwIHN0YXRlLCByZWFsLW1vZGVsIGZyZXNoLXByb2Nlc3Mgc2VhbSB0ZXN0LiIiIgpmcm9tIFBJTCBpbXBvcnQgSW1hZ2UKaW1wb3J0IGFyZ3BhcnNlLGNvcHksanNvbixtYXRoLG9zLHJhbmRvbSxzaWduYWwsc3VicHJvY2VzcyxzeXMsdGltZSx0cmFjZWJhY2sKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSBmaWxlbG9jayBpbXBvcnQgRmlsZUxvY2sKaW1wb3J0IHBoYXNlMl90cmFpbmluZ19kYXRhIGFzIGQKaW1wb3J0IHBoYXNlMl9tb2RlbHMgYXMgbQoKU1RPUD1GYWxzZQpTVEVQX1NFQ09ORFM9W10KZGVmIGVudmlyb25tZW50KCk6CiBpbXBvcnQgaW1wb3J0bGliLm1ldGFkYXRhCiByZXR1cm4gZGljdChwYWNrYWdlcz17azppbXBvcnRsaWIubWV0YWRhdGEudmVyc2lvbihrKSBmb3IgayBpbiAoJ3RvcmNoJywndG9yY2h2aXNpb24nLCdudW1weScsJ1BpbGxvdycsJ3RpbW0nLCd0cmFuc2Zvcm1lcnMnLCd1bHRyYWx5dGljcycpfSxncHU9dG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoMCkgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICdDUFUnKQpkZWYgc3RvcCgqXyk6CiBnbG9iYWwgU1RPUAogU1RPUD1UcnVlCmRlZiBzZWVkX2FsbChzZWVkKTpyYW5kb20uc2VlZChzZWVkKTtucC5yYW5kb20uc2VlZChzZWVkKTt0b3JjaC5tYW51YWxfc2VlZChzZWVkKTt0b3JjaC5jdWRhLm1hbnVhbF9zZWVkX2FsbChzZWVkKQpkZWYgcm5nKCk6cmV0dXJuIGRpY3QocHl0aG9uPXJhbmRvbS5nZXRzdGF0ZSgpLG51bXB5PW5wLnJhbmRvbS5nZXRfc3RhdGUoKSx0b3JjaD10b3JjaC5nZXRfcm5nX3N0YXRlKCksY3VkYT10b3JjaC5jdWRhLmdldF9ybmdfc3RhdGVfYWxsKCkgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlIFtdKQpkZWYgc2V0X3JuZyhzKToKIHJhbmRvbS5zZXRzdGF0ZShzWydweXRob24nXSk7bnAucmFuZG9tLnNldF9zdGF0ZShzWydudW1weSddKTt0b3JjaC5zZXRfcm5nX3N0YXRlKHNbJ3RvcmNoJ10pCiBpZiBzWydjdWRhJ106dG9yY2guY3VkYS5zZXRfcm5nX3N0YXRlX2FsbChzWydjdWRhJ10pCmRlZiBjcHUoeCk6CiBpZiBpc2luc3RhbmNlKHgsdG9yY2guVGVuc29yKTpyZXR1cm4geC5kZXRhY2goKS5jcHUoKS5jbG9uZSgpCiBpZiBpc2luc3RhbmNlKHgsZGljdCk6cmV0dXJuIHtrOmNwdSh2KSBmb3Igayx2IGluIHguaXRlbXMoKX0KIGlmIGlzaW5zdGFuY2UoeCxsaXN0KTpyZXR1cm4gW2NwdSh2KSBmb3IgdiBpbiB4XQogaWYgaXNpbnN0YW5jZSh4LHR1cGxlKTpyZXR1cm4gdHVwbGUoY3B1KHYpIGZvciB2IGluIHgpCiByZXR1cm4gY29weS5kZWVwY29weSh4KQpkZWYgZGVsdGEoYSxiKToKIGlmIGlzaW5zdGFuY2UoYSx0b3JjaC5UZW5zb3IpOgogIGFzc2VydCBhLnNoYXBlPT1iLnNoYXBlIGFuZCBhLmR0eXBlPT1iLmR0eXBlCiAgcmV0dXJuIGZsb2F0KChhLmRvdWJsZSgpLWIuZG91YmxlKCkpLmFicygpLm1heCgpKSBpZiBhLm51bWVsKCkgZWxzZSAwLgogaWYgaXNpbnN0YW5jZShhLG5wLm5kYXJyYXkpOnJldHVybiBmbG9hdChucC5hYnMoYS5hc3R5cGUoZmxvYXQpLWIuYXN0eXBlKGZsb2F0KSkubWF4KCkpCiBpZiBpc2luc3RhbmNlKGEsZGljdCk6CiAgYXNzZXJ0IGEua2V5cygpPT1iLmtleXMoKTtyZXR1cm4gbWF4KFtkZWx0YShhW2tdLGJba10pIGZvciBrIGluIGFdIG9yIFswLl0pCiBpZiBpc2luc3RhbmNlKGEsKHR1cGxlLGxpc3QpKToKICBhc3NlcnQgbGVuKGEpPT1sZW4oYik7cmV0dXJuIG1heChbZGVsdGEoeCx5KSBmb3IgeCx5IGluIHppcChhLGIpXSBvciBbMC5dKQogaWYgaXNpbnN0YW5jZShhLChmbG9hdCxpbnQpKTpyZXR1cm4gYWJzKGEtYikKIGFzc2VydCBhPT1iO3JldHVybiAwLgoKZGVmIGNvbXBvbmVudHMoam9iLGFzc2V0cyxkZXZpY2UsaW5pdGlhbGl6ZSk6CiBtb2RlbD1tLmJ1aWxkKGpvYlsnbW9kZWwnXSxhc3NldHMsaW5pdGlhbGl6ZSxkZXZpY2UpCiBvcHQ9dG9yY2gub3B0aW0uQWRhbVcobW9kZWwucGFyYW1ldGVycygpLGxyPTFlLTQsd2VpZ2h0X2RlY2F5PS4wMSkKIHNjYWxlcj10b3JjaC5hbXAuR3JhZFNjYWxlcihkZXZpY2UudHlwZSxlbmFibGVkPWRldmljZS50eXBlPT0nY3VkYScsaW5pdF9zY2FsZT0xMDI0LikKIHNjaGVkPXRvcmNoLm9wdGltLmxyX3NjaGVkdWxlci5Db3NpbmVBbm5lYWxpbmdMUihvcHQsVF9tYXg9NjAsZXRhX21pbj0xZS02KQogcmV0dXJuIG1vZGVsLG9wdCxzY2FsZXIsc2NoZWQKCmRlZiB1cGRhdGUobW9kZWwsb3B0LHNjYWxlcixiLGpvYixlcG9jaCk6CiBzdGFydD1ybmcoKTtidWZmZXJzPXtrOnYuZGV0YWNoKCkuY2xvbmUoKSBmb3Igayx2IGluIG1vZGVsLm5hbWVkX2J1ZmZlcnMoKX07ZXZlbnRzPVtdCiBmb3IgYXR0ZW1wdCBpbiByYW5nZSg5KToKICBzZXRfcm5nKHN0YXJ0KQogIGZvciBrLHYgaW4gbW9kZWwubmFtZWRfYnVmZmVycygpOnYuY29weV8oYnVmZmVyc1trXSkKICBtLnRyYWluaW5nX21vZGUobW9kZWwpO29wdC56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICB3aXRoIHRvcmNoLmF1dG9jYXN0KGJbJ2ltZyddLmRldmljZS50eXBlLGR0eXBlPXRvcmNoLmZsb2F0MTYsZW5hYmxlZD1iWydpbWcnXS5pc19jdWRhIGFuZCBhdHRlbXB0PDgpOnZhbHVlPW0ubG9zcyhtb2RlbCxiLGpvYlsnbW9kZWwnXSxlcG9jaCkKICBzY2FsZWQ9c2NhbGVyLnNjYWxlKHZhbHVlKTtmaW5pdGU9Ym9vbCh0b3JjaC5pc2Zpbml0ZSh2YWx1ZSkpCiAgaWYgZmluaXRlOgogICBzY2FsZWQuYmFja3dhcmQoKTtzY2FsZXIudW5zY2FsZV8ob3B0KQogICBub3JtPXRvcmNoLm5uLnV0aWxzLmNsaXBfZ3JhZF9ub3JtXyhtb2RlbC5wYXJhbWV0ZXJzKCksMS4pO2Zpbml0ZT1ib29sKHRvcmNoLmlzZmluaXRlKG5vcm0pKQogIGlmIGZpbml0ZToKICAgc2NhbGVyLnN0ZXAob3B0KTtzY2FsZXIudXBkYXRlKCk7cmV0dXJuIGZsb2F0KHZhbHVlLmRldGFjaCgpKSxldmVudHMKICBvcHQuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpCiAgaWYgYXR0ZW1wdD09OCBvciBub3Qgc2NhbGVyLmlzX2VuYWJsZWQoKToKICAgc2V0X3JuZyhzdGFydCk7cmFpc2UgUnVudGltZUVycm9yKCdOb25maW5pdGUgdXBkYXRlOiBwcmV2aW91cyBkdXJhYmxlIGNoZWNrcG9pbnQgcmV0YWluZWQ7IG5vIHNhbXBsZSBza2lwcGVkLicpCiAgc2NhbGU9ZmxvYXQoc2NhbGVyLmdldF9zY2FsZSgpKTtzY2FsZXIudXBkYXRlKG5ld19zY2FsZT1tYXgoc2NhbGUqLjUsMWUtOCkpO2V2ZW50cy5hcHBlbmQoZGljdChhdHRlbXB0PWF0dGVtcHQrMSxzY2FsZT1zY2FsZSxzdGF0dXM9J3JldHJ5X3NhbWVfYmF0Y2gnKSkKIHJhaXNlIEFzc2VydGlvbkVycm9yKCd1bnJlYWNoYWJsZScpCgpkZWYgc3RhdGUobW9kZWwsb3B0LHNjYWxlcixzY2hlZCxqb2IscHJvdG9jb2wsZXBvY2gsY3Vyc29yLGhpc3RvcnksdmFsaWRhdGlvbixiZXN0LGJlc3Rfc2NvcmUsdXBkYXRlcyk6CiByZXR1cm4gZGljdChtb2RlbD1tb2RlbC5zdGF0ZV9kaWN0KCksb3B0aW1pemVyPW9wdC5zdGF0ZV9kaWN0KCksc2NhbGVyPXNjYWxlci5zdGF0ZV9kaWN0KCksc2NoZWR1bGVyPXNjaGVkLnN0YXRlX2RpY3QoKSxybmc9cm5nKCksam9iPWpvYixwcm90b2NvbD1wcm90b2NvbCx0b3JjaF92ZXJzaW9uPXRvcmNoLl9fdmVyc2lvbl9fLGVudmlyb25tZW50PWVudmlyb25tZW50KCksZXBvY2g9ZXBvY2gsY3Vyc29yPWN1cnNvcixoaXN0b3J5PWhpc3RvcnksdmFsaWRhdGlvbj12YWxpZGF0aW9uLGJlc3Q9YmVzdCxiZXN0X3Njb3JlPWJlc3Rfc2NvcmUsdXBkYXRlcz11cGRhdGVzKQpkZWYgc2F2ZShmb2xkZXIscyxzdGF0dXM9J3Jlc3VtYWJsZScpOgogZm9sZGVyPVBhdGgoZm9sZGVyKTtmb2xkZXIubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiB3aXRoIEZpbGVMb2NrKHN0cihmb2xkZXIvJ2NoZWNrcG9pbnQubG9jaycpKToKICB0bXA9Zm9sZGVyLydzdGF0ZS50bXAnCiAgd2l0aCBvcGVuKHRtcCwnd2InKSBhcyBmOgogICB0b3JjaC5zYXZlKHMsZik7Zi5mbHVzaCgpO29zLmZzeW5jKGYuZmlsZW5vKCkpCiAgb3MucmVwbGFjZSh0bXAsZm9sZGVyLydzdGF0ZS5wdCcpCiAgZC53cml0ZShmb2xkZXIvJ0xPQ0FMLmpzb24nLGRpY3Qoc3RhdHVzPXN0YXR1cyxlcG9jaD1zWydlcG9jaCddLGN1cnNvcj1zWydjdXJzb3InXSx1cGRhdGVzPXNbJ3VwZGF0ZXMnXSxwcm90b2NvbD1zWydwcm90b2NvbCddLGpvYj1zWydqb2InXSkpCmRlZiByZXN0b3JlKHMsbW9kZWwsb3B0LHNjYWxlcixzY2hlZCxwcm90b2NvbCxqb2IpOgogYXNzZXJ0IHNbJ3Byb3RvY29sJ109PXByb3RvY29sIGFuZCBzWydqb2InXT09am9iLCdJbmNvbXBhdGlibGUgY2hlY2twb2ludDogbmV2ZXIgc2lsZW50bHkgcmVzdGFydCcKIGFzc2VydCBzWyd0b3JjaF92ZXJzaW9uJ109PXRvcmNoLl9fdmVyc2lvbl9fLCdUb3JjaCBjaGFuZ2VkIHNpbmNlIHRoZSBjaGVja3BvaW50OyBuZWVkcyBhbiBleHBsaWNpdCBjb21wYXRpYmlsaXR5IGNoZWNrLCBuZXZlciByZXN0YXJ0IHNpbGVudGx5JwogYXNzZXJ0IHNbJ2Vudmlyb25tZW50J109PWVudmlyb25tZW50KCksJ0NoZWNrcG9pbnQgcnVudGltZS9HUFUgZGlmZmVyczsgZXhwbGljaXQgY29tcGF0aWJpbGl0eSByZXZpZXcgcmVxdWlyZWQsIG5vIHNpbGVudCByZXN0YXJ0JwogbW9kZWwubG9hZF9zdGF0ZV9kaWN0KHNbJ21vZGVsJ10sc3RyaWN0PVRydWUpO29wdC5sb2FkX3N0YXRlX2RpY3Qoc1snb3B0aW1pemVyJ10pO3NjYWxlci5sb2FkX3N0YXRlX2RpY3Qoc1snc2NhbGVyJ10pO3NjaGVkLmxvYWRfc3RhdGVfZGljdChzWydzY2hlZHVsZXInXSk7c2V0X3JuZyhzWydybmcnXSkKCmRlZiBldmFsdWF0ZShtb2RlbCxyb3dzLHJvb3Qsam9iLGRldmljZSk6CiBuYW1lPWpvYlsnbW9kZWwnXTttb2RlbC5ldmFsKCk7cmVjb3Jkcz1bXQogd2l0aCB0b3JjaC5pbmZlcmVuY2VfbW9kZSgpOgogIGZvciByb3cgaW4gcm93czoKICAgaWYgU1RPUDpyYWlzZSBJbnRlcnJ1cHRlZEVycm9yKCdFdmFsdWF0aW9uIHBhdXNlZDsgdHJhaW5pbmcgc3RhdGUgaXMgZHVyYWJsZScpCiAgIGlmIG5hbWU9PSdocm5ldCcgYW5kIHJvd1sncG9pbnRfa2luZCddIT0naHVtYW4nOmNvbnRpbnVlCiAgIGI9bS5iYXRjaChyb290LFtyb3ddLG5hbWUsZGV2aWNlPWRldmljZSk7dD10aW1lLnBlcmZfY291bnRlcigpCiAgIHdpdGggdG9yY2guYXV0b2Nhc3QoZGV2aWNlLnR5cGUsZHR5cGU9dG9yY2guZmxvYXQxNixlbmFibGVkPWRldmljZS50eXBlPT0nY3VkYScpOnByZWQ9bS5wcmVkaWN0KG1vZGVsLGIsbmFtZSkKICAgaWYgZGV2aWNlLnR5cGU9PSdjdWRhJzp0b3JjaC5jdWRhLnN5bmNocm9uaXplKCkKICAgcmVjPWRpY3QoaW1hZ2VfaWQ9cm93WydpbWFnZV9pZCddLHR5cmU9cm93WydwaHlzaWNhbF90eXJlX2lkJ10sZG9tYWluPXJvd1snZG9tYWluJ10sc2Vjb25kcz10aW1lLnBlcmZfY291bnRlcigpLXQpCiAgIGlmIG5hbWUgaW4gKCdtb2JpbGVuZXR2NCcsJ3Jlc25ldDUwJyk6CiAgICByZWMudXBkYXRlKGxhYmVsPXJvd1snY2xhc3NfaW5kZXgnXSxwcmVkaWN0aW9uPWludChwcmVkWzBdLmFyZ21heCgpKSxwcm9iYWJpbGl0aWVzPXByZWRbMF0uY3B1KCkudG9saXN0KCkpCiAgIGVsaWYgbmFtZT09J2hybmV0JzoKICAgIHJlYy51cGRhdGUodGFyZ2V0PXJvd1sncG9pbnRzJ10scHJlZGljdGlvbj1wcmVkWzBdLmNwdSgpLnRvbGlzdCgpLG1hZT1mbG9hdCgocHJlZFswXS1iWydwb2ludHMnXVswXSkuYWJzKCkubWVhbigpKSkKICAgZWxzZToKICAgICMgU2NvcmUgb3JpZ2luYWwgcmF3IG1hc2tzLCBleGNsdWRpbmcgY29udHJhZGljdGlvbiBwaXhlbHMsIGF0IG1vZGVsIGlucHV0IHJlc29sdXRpb24uCiAgICBfLHJhdyxfPWQuc2FtcGxlKHJvb3Qscm93LGQuU1BFQ1NbbmFtZV1bJ2h3J10pO3RhcmdldD10b3JjaC50ZW5zb3IocmF3WzoyXSxkZXZpY2U9ZGV2aWNlKTt2YWxpZD10b3JjaC50ZW5zb3IofnJhd1syXSxkZXZpY2U9ZGV2aWNlKQogICAgdmFsdWVzPVtdCiAgICBmb3IgYyBpbiAoMCwxKToKICAgICBwPXByZWRbMCxjXSZ2YWxpZDt5PXRhcmdldFtjXSZ2YWxpZDtpbnRlcj1pbnQoKHAmeSkuc3VtKCkpO2Rlbj1pbnQocC5zdW0oKSt5LnN1bSgpKTt1bmlvbj1pbnQoKHB8eSkuc3VtKCkpCiAgICAgdmFsdWVzLmFwcGVuZChkaWN0KGRpY2U9KDIqaW50ZXIrMSkvKGRlbisxKSxpb3U9KGludGVyKzEpLyh1bmlvbisxKSxwcmVkaWN0ZWRfcGl4ZWxzPWludChwLnN1bSgpKSx0YXJnZXRfcGl4ZWxzPWludCh5LnN1bSgpKSkpCiAgICByZWNbJ3JlZ2lvbnMnXT12YWx1ZXMKICAgcmVjb3Jkcy5hcHBlbmQocmVjKQogYXNzZXJ0IHJlY29yZHMKIGlmIG5hbWUgaW4gKCdtb2JpbGVuZXR2NCcsJ3Jlc25ldDUwJyk6CiAgcmVjYWxscz17c3RyKGMpOnN1bShyWydwcmVkaWN0aW9uJ109PWMgZm9yIHIgaW4gcmVjb3JkcyBpZiByWydsYWJlbCddPT1jKS9zdW0oclsnbGFiZWwnXT09YyBmb3IgciBpbiByZWNvcmRzKSBmb3IgYyBpbiBzb3J0ZWQoe3JbJ2xhYmVsJ10gZm9yIHIgaW4gcmVjb3Jkc30pfQogIHNjb3JlPWZsb2F0KG5wLm1lYW4obGlzdChyZWNhbGxzLnZhbHVlcygpKSkpO21ldHJpYz0nYmFsYW5jZWRfYWNjdXJhY3lfb25fb2JzZXJ2ZWRfbG93X2hpZ2hfY2xhc3NlcycKIGVsaWYgbmFtZT09J2hybmV0JzpzY29yZT0tZmxvYXQobnAubWVhbihbclsnbWFlJ10gZm9yIHIgaW4gcmVjb3Jkc10pKTttZXRyaWM9J25lZ2F0aXZlX21lYW5fd2lkdGhfZnJhY3Rpb25fZXJyb3InCiBlbHNlOnNjb3JlPWZsb2F0KG5wLm1lYW4oW3hbJ2RpY2UnXSBmb3IgciBpbiByZWNvcmRzIGZvciB4IGluIHJbJ3JlZ2lvbnMnXV0pKTttZXRyaWM9J21lYW5fdHdvX3JlZ2lvbl9kaWNlX2F0X2lucHV0X3Jlc29sdXRpb24nCiByZXR1cm4gZGljdChtZXRyaWM9bWV0cmljLHNjb3JlPXNjb3JlLHJlY29yZHM9cmVjb3JkcyxwZXJfdHlyZT17dHlyZTpzdW0oclsndHlyZSddPT10eXJlIGZvciByIGluIHJlY29yZHMpIGZvciB0eXJlIGluIHNvcnRlZCh7clsndHlyZSddIGZvciByIGluIHJlY29yZHN9KX0sc2NvcGU9J2tub3duIGhlbGQtb3V0IG9sZCBsb3cvaGlnaCB0eXJlcyBvbmx5OyBub3QgdW5zZWVuIG1pZC92aWRlbyBldmlkZW5jZScpCgpkZWYgcnVuX3N0ZXBzKHMsbW9kZWwsb3B0LHNjYWxlcixzY2hlZCxwbGFuLHJvb3Qsam9iLGRldmljZSxuKToKIHJvd3MsYmF0Y2hlcz1kLnNjaGVkdWxlKHBsYW4sam9iLDApCiBsb3NzZXM9W10KIGZvciBzdGVwIGluIHJhbmdlKHNbJ2N1cnNvciddLHNbJ2N1cnNvciddK24pOgogIHN0YXJ0ZWQ9dGltZS5tb25vdG9uaWMoKQogIGluZGljZXM9YmF0Y2hlc1tzdGVwJWxlbihiYXRjaGVzKV0KICBiPW0uYmF0Y2gocm9vdCxbcm93c1tpXSBmb3IgaSBpbiBpbmRpY2VzXSxqb2JbJ21vZGVsJ10sMCxzdGVwLGpvYlsnc2VlZCddLGRldmljZSkKICB2YWx1ZSxfPXVwZGF0ZShtb2RlbCxvcHQsc2NhbGVyLGIsam9iLDApO2xvc3Nlcy5hcHBlbmQodmFsdWUpCiAgaWYgZGV2aWNlLnR5cGU9PSdjdWRhJzp0b3JjaC5jdWRhLnN5bmNocm9uaXplKCkKICBTVEVQX1NFQ09ORFMuYXBwZW5kKHRpbWUubW9ub3RvbmljKCktc3RhcnRlZCkKIHJldHVybiBsb3NzZXMKCmRlZiBzbW9rZShjb25maWcsZm9sZGVyLGRldmljZSxjb250aW51YXRpb249RmFsc2UpOgogcGxhbj1jb25maWdbJ3BsYW4nXTtqb2I9Y29uZmlnWydqb2InXTtwcm90b2NvbD1jb25maWdbJ3Byb3RvY29sJ107cm9vdD1jb25maWdbJ3Jvb3QnXTtmb2xkZXI9UGF0aChmb2xkZXIpO2ZvbGRlci5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKIHNlZWRfYWxsKGpvYlsnc2VlZCddKTttb2RlbCxvcHQsc2NhbGVyLHNjaGVkPWNvbXBvbmVudHMoam9iLGNvbmZpZ1snYXNzZXRzJ10sZGV2aWNlLEZhbHNlIGlmIGNvbnRpbnVhdGlvbiBlbHNlIFRydWUpCiBpZiBjb250aW51YXRpb246CiAgcz10b3JjaC5sb2FkKGZvbGRlci8nc2VhbS5wdCcsbWFwX2xvY2F0aW9uPSdjcHUnLHdlaWdodHNfb25seT1GYWxzZSk7cmVzdG9yZShzLG1vZGVsLG9wdCxzY2FsZXIsc2NoZWQscHJvdG9jb2wsam9iKQogIGxvc3Nlcz1ydW5fc3RlcHMocyxtb2RlbCxvcHQsc2NhbGVyLHNjaGVkLHBsYW4scm9vdCxqb2IsZGV2aWNlLDIpCiAgdG9yY2guc2F2ZShkaWN0KG1vZGVsPWNwdShtb2RlbC5zdGF0ZV9kaWN0KCkpLG9wdGltaXplcj1jcHUob3B0LnN0YXRlX2RpY3QoKSksc2NhbGVyPXNjYWxlci5zdGF0ZV9kaWN0KCksc2NoZWR1bGVyPXNjaGVkLnN0YXRlX2RpY3QoKSxybmc9cm5nKCksbG9zc2VzPWxvc3NlcyksZm9sZGVyLydhY3R1YWwucHQnKTtyZXR1cm4KIHN0YXJ0PXRpbWUubW9ub3RvbmljKCk7aW5pdGlhbD1kaWN0KGN1cnNvcj0wKQogcnVuX3N0ZXBzKGluaXRpYWwsbW9kZWwsb3B0LHNjYWxlcixzY2hlZCxwbGFuLHJvb3Qsam9iLGRldmljZSwyKQogIyBFeGVyY2lzZSBhIG5vbi1pbml0aWFsIHNjaGVkdWxlciBzdGF0ZSBhcyB3ZWxsIGFzIG9wdGltaXplci9zY2FsZXIgc3RhdGUuCiBzY2hlZC5zdGVwKCkKIHRvcmNoLnNhdmUoc3RhdGUobW9kZWwsb3B0LHNjYWxlcixzY2hlZCxqb2IscHJvdG9jb2wsMCwyLFtdLFtdLE5vbmUsLW1hdGguaW5mLDIpLGZvbGRlci8nc2VhbS5wdCcpCiBsb3NzZXM9cnVuX3N0ZXBzKGRpY3QoY3Vyc29yPTIpLG1vZGVsLG9wdCxzY2FsZXIsc2NoZWQscGxhbixyb290LGpvYixkZXZpY2UsMikKIGV4cGVjdGVkPWRpY3QobW9kZWw9Y3B1KG1vZGVsLnN0YXRlX2RpY3QoKSksb3B0aW1pemVyPWNwdShvcHQuc3RhdGVfZGljdCgpKSxzY2FsZXI9c2NhbGVyLnN0YXRlX2RpY3QoKSxzY2hlZHVsZXI9c2NoZWQuc3RhdGVfZGljdCgpLHJuZz1ybmcoKSxsb3NzZXM9bG9zc2VzKQogIyBSZWxlYXNlIFZSQU0gYmVmb3JlIGluZGVwZW5kZW50IHByb2Nlc3MgcmVjb25zdHJ1Y3RzIGFuZCByZXN0b3JlcyB0aGUgcmVhbCBtb2RlbC4KIGRlbCBtb2RlbCxvcHQsc2NhbGVyLHNjaGVkO3RvcmNoLmN1ZGEuZW1wdHlfY2FjaGUoKQogYXJncz1bc3lzLmV4ZWN1dGFibGUsJy1CJyxfX2ZpbGVfXywnY29udGludWUtc21va2UnLCctLWNvbmZpZycsc3RyKGZvbGRlci8nY29uZmlnLmpzb24nKSwnLS1mb2xkZXInLHN0cihmb2xkZXIpLCctLWRldmljZScsc3RyKGRldmljZSldCiBkLndyaXRlKGZvbGRlci8nY29uZmlnLmpzb24nLGNvbmZpZyk7c3VicHJvY2Vzcy5ydW4oYXJncyxjaGVjaz1UcnVlKQogYWN0dWFsPXRvcmNoLmxvYWQoZm9sZGVyLydhY3R1YWwucHQnLG1hcF9sb2NhdGlvbj0nY3B1Jyx3ZWlnaHRzX29ubHk9RmFsc2UpCiBkaWZmZXJlbmNlcz17azpkZWx0YShleHBlY3RlZFtrXSxhY3R1YWxba10pIGZvciBrIGluIGV4cGVjdGVkfQogcGFzc2VkPW1heChkaWZmZXJlbmNlcy52YWx1ZXMoKSk8PTFlLTUKIHJlcG9ydD1kaWN0KHN0YXR1cz0ncGFzc2VkJyBpZiBwYXNzZWQgZWxzZSAnZmFpbGVkJyxwcm90b2NvbD1wcm90b2NvbCxqb2I9am9iLGRpZmZlcmVuY2VzPWRpZmZlcmVuY2VzLHNlY29uZHM9dGltZS5tb25vdG9uaWMoKS1zdGFydCx0b3JjaD10b3JjaC5fX3ZlcnNpb25fXyxncHU9dG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoZGV2aWNlKSBpZiBkZXZpY2UudHlwZT09J2N1ZGEnIGVsc2UgJ0NQVScscGVha19ncHVfYnl0ZXM9dG9yY2guY3VkYS5tYXhfbWVtb3J5X2FsbG9jYXRlZCgpIGlmIGRldmljZS50eXBlPT0nY3VkYScgZWxzZSBOb25lLHNvdXJjZV9zaGEyNTY9ZC5zaGEoX19maWxlX18pLHNjb3BlPSdyZWFsIG1vZGVsLCByZWFsIHRyYW5zZm9ybWVkIGJhdGNoZXM7IGluZGVwZW5kZW50LXByb2Nlc3MgbWlkLWVwb2NoIHN0YXRlIHJlc3RvcmF0aW9uOyBub25pbml0aWFsIHNjaGVkdWxlciBzdGF0ZScpCiBzdGVwcz1sZW4oZC5zY2hlZHVsZShwbGFuLGpvYiwwKVsxXSk7cmVwb3J0WydtZWFzdXJlZF9zdGVwX3NlY29uZHMnXT1TVEVQX1NFQ09ORFNbLTQ6XQogcmVwb3J0Wydlc3RpbWF0ZWRfdHJhaW5pbmdfY29tcHV0ZV9ob3Vyc19leGNsdWRpbmdfY2hlY2twb2ludF92YWxpZGF0aW9uX3VwbG9hZCddPWZsb2F0KG5wLm1lYW4oU1RFUF9TRUNPTkRTWy00Ol0pKSpzdGVwcyo2MC8zNjAwCiBkLndyaXRlKGZvbGRlci8nU01PS0UuanNvbicscmVwb3J0KQogZm9yIG5hbWUgaW4gKCdzZWFtLnB0JywnYWN0dWFsLnB0Jyk6KGZvbGRlci9uYW1lKS51bmxpbmsoKQogaWYgbm90IHBhc3NlZDpyYWlzZSBSdW50aW1lRXJyb3IoJ1JlYWwtbW9kZWwgZnJlc2gtcHJvY2VzcyByZXN1bWUgdGVzdCBmYWlsZWQ7IGxvbmcgcnVuIHdhcyBub3Qgc3RhcnRlZCcpCiBwcmludCgnR1BVIHJlc3VtZSBwYXNzZWQuIEVzdGltYXRlZCB0cmFpbmluZyBjb21wdXRlIGhvdXJzIGZvciB0aGlzIHJ1biAoZXhjbHVkZXMgY2hlY2twb2ludCwgdmFsaWRhdGlvbiBhbmQgdXBsb2Fkcyk6Jyxyb3VuZChyZXBvcnRbJ2VzdGltYXRlZF90cmFpbmluZ19jb21wdXRlX2hvdXJzX2V4Y2x1ZGluZ19jaGVja3BvaW50X3ZhbGlkYXRpb25fdXBsb2FkJ10sMiksZmx1c2g9VHJ1ZSkKIHJldHVybiByZXBvcnQKCmRlZiB0cmFpbihjb25maWcsZm9sZGVyLGRldmljZSk6CiBwbGFuPWNvbmZpZ1sncGxhbiddO2pvYj1jb25maWdbJ2pvYiddO3Byb3RvY29sPWNvbmZpZ1sncHJvdG9jb2wnXTtyb290PWNvbmZpZ1sncm9vdCddO2ZvbGRlcj1QYXRoKGZvbGRlcikKIHNtb2tlX2Rpcj1mb2xkZXIvJ3Ntb2tlJztyZXN1bHQ9c21va2UoY29uZmlnLHNtb2tlX2RpcixkZXZpY2UpCiBpZiBTVE9QOnJldHVybgogZC53cml0ZShmb2xkZXIvJ1NNT0tFLmpzb24nLHJlc3VsdCkKIHNlZWRfYWxsKGpvYlsnc2VlZCddKTtleGlzdGluZz0oZm9sZGVyLydzdGF0ZS5wdCcpLmV4aXN0cygpCiBtb2RlbCxvcHQsc2NhbGVyLHNjaGVkPWNvbXBvbmVudHMoam9iLGNvbmZpZ1snYXNzZXRzJ10sZGV2aWNlLG5vdCBleGlzdGluZykKIGlkZW50aXR5PWRpY3QobW9kZWw9ZC5TUEVDU1tqb2JbJ21vZGVsJ11dLHBhcmFtZXRlcnM9c3VtKHAubnVtZWwoKSBmb3IgcCBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkpLHRlbnNvcl9zaWduYXR1cmU9ZC5kaWdlc3Qoe2s6bGlzdCh2LnNoYXBlKSBmb3Igayx2IGluIG1vZGVsLnN0YXRlX2RpY3QoKS5pdGVtcygpfSkpCiBkLndyaXRlKGZvbGRlci8nSURFTlRJVFkuanNvbicsaWRlbnRpdHkpCiBpZiBoYXNhdHRyKG1vZGVsLCdjb25maWcnKTpkLndyaXRlKGZvbGRlci8nTU9ERUxfQ09ORklHLmpzb24nLG1vZGVsLmNvbmZpZy50b19kaWN0KCkpCiBlbGlmIGhhc2F0dHIobW9kZWwsJ3lhbWwnKTpkLndyaXRlKGZvbGRlci8nTU9ERUxfQ09ORklHLmpzb24nLG1vZGVsLnlhbWwpCiBpZiBleGlzdGluZzoKICBzPXRvcmNoLmxvYWQoZm9sZGVyLydzdGF0ZS5wdCcsbWFwX2xvY2F0aW9uPSdjcHUnLHdlaWdodHNfb25seT1GYWxzZSk7cmVzdG9yZShzLG1vZGVsLG9wdCxzY2FsZXIsc2NoZWQscHJvdG9jb2wsam9iKQogZWxzZTpzPXN0YXRlKG1vZGVsLG9wdCxzY2FsZXIsc2NoZWQsam9iLHByb3RvY29sLDAsMCxbXSxbXSxOb25lLC1tYXRoLmluZiwwKTtzYXZlKGZvbGRlcixzKQogZXBvY2g9c1snZXBvY2gnXTtjdXJzb3I9c1snY3Vyc29yJ107aGlzdG9yeT1zWydoaXN0b3J5J107dmFsaWRhdGlvbj1zWyd2YWxpZGF0aW9uJ107YmVzdD1zWydiZXN0J107YmVzdF9zY29yZT1zWydiZXN0X3Njb3JlJ107dXBkYXRlcz1zWyd1cGRhdGVzJ107ZGVsIHMKIHN0YXJ0ZWQ9dGltZS5tb25vdG9uaWMoKQogdHJ5OgogIHdoaWxlIGVwb2NoPDYwOgogICByb3dzLGJhdGNoZXM9ZC5zY2hlZHVsZShwbGFuLGpvYixlcG9jaCkKICAgd2hpbGUgY3Vyc29yPGxlbihiYXRjaGVzKToKICAgIGlmIFNUT1Agb3IgdGltZS5tb25vdG9uaWMoKS1zdGFydGVkPjguNSozNjAwOnJldHVybgogICAgaWYgX19pbXBvcnRfXygnc2h1dGlsJykuZGlza191c2FnZShmb2xkZXIpLmZyZWU8MioxMDI0KiozOnJhaXNlIFJ1bnRpbWVFcnJvcignTGVzcyB0aGFuIDIgR2lCIGZyZWU7IHBhdXNlIGJlZm9yZSBhbiB1bnNhZmUgY2hlY2twb2ludCB3cml0ZScpCiAgICBpZHM9YmF0Y2hlc1tjdXJzb3JdO2I9bS5iYXRjaChyb290LFtyb3dzW2ldIGZvciBpIGluIGlkc10sam9iWydtb2RlbCddLGVwb2NoLGN1cnNvcixqb2JbJ3NlZWQnXSxkZXZpY2UpCiAgICB0PXRpbWUubW9ub3RvbmljKCk7dmFsdWUsZXZlbnRzPXVwZGF0ZShtb2RlbCxvcHQsc2NhbGVyLGIsam9iLGVwb2NoKTtjdXJzb3IrPTE7dXBkYXRlcys9MQogICAgaGlzdG9yeS5hcHBlbmQoZGljdChlcG9jaD1lcG9jaCxiYXRjaD1jdXJzb3IsbG9zcz12YWx1ZSxscj1vcHQucGFyYW1fZ3JvdXBzWzBdWydsciddLHNlY29uZHM9dGltZS5tb25vdG9uaWMoKS10LGltYWdlX2lkcz1bcm93c1tpXVsnaW1hZ2VfaWQnXSBmb3IgaSBpbiBpZHNdLG51bWVyaWNhbF9yZXRyaWVzPWV2ZW50cykpCiAgICBzYXZlKGZvbGRlcixzdGF0ZShtb2RlbCxvcHQsc2NhbGVyLHNjaGVkLGpvYixwcm90b2NvbCxlcG9jaCxjdXJzb3IsaGlzdG9yeSx2YWxpZGF0aW9uLGJlc3QsYmVzdF9zY29yZSx1cGRhdGVzKSk7ZGVsIGIKICAgdmFsPWV2YWx1YXRlKG1vZGVsLFtyIGZvciByIGluIHBsYW5bJ3Jvd3MnXSBpZiByWydyb2xlJ109PSd2YWxpZGF0aW9uJ10scm9vdCxqb2IsZGV2aWNlKQogICB2YWxpZGF0aW9uLmFwcGVuZChkaWN0KGVwb2NoPWVwb2NoKzEscmVzdWx0PXZhbCkpCiAgIGlmIHZhbFsnc2NvcmUnXT5iZXN0X3Njb3JlOmJlc3Q9Y3B1KG1vZGVsLnN0YXRlX2RpY3QoKSk7YmVzdF9zY29yZT12YWxbJ3Njb3JlJ10KICAgc2NoZWQuc3RlcCgpO2Vwb2NoKz0xO2N1cnNvcj0wCiAgIHNhdmUoZm9sZGVyLHN0YXRlKG1vZGVsLG9wdCxzY2FsZXIsc2NoZWQsam9iLHByb3RvY29sLGVwb2NoLGN1cnNvcixoaXN0b3J5LHZhbGlkYXRpb24sYmVzdCxiZXN0X3Njb3JlLHVwZGF0ZXMpKQogICBwcmludChmIntqb2JbJ2lkJ119IGVwb2NoIHtlcG9jaH0vNjAgfCB2YWxpZGF0aW9uIHt2YWxbJ21ldHJpYyddfSB7dmFsWydzY29yZSddOi41Zn0iLGZsdXNoPVRydWUpCiAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KGJlc3Qsc3RyaWN0PVRydWUpCiAgdGVzdD1ldmFsdWF0ZShtb2RlbCxbciBmb3IgciBpbiBwbGFuWydyb3dzJ10gaWYgclsncm9sZSddPT0ndGVzdCddLHJvb3Qsam9iLGRldmljZSkKICBkLndyaXRlKGZvbGRlci8nRklOQUwuanNvbicsZGljdChwcm90b2NvbD1wcm90b2NvbCxqb2I9am9iLHZhbGlkYXRpb25fc2VsZWN0aW9uPSdoaWdoZXN0IHZhbGlkYXRpb24gc2NvcmU7IGVhcmxpZXN0IHRpZScsYmVzdF9zY29yZT1iZXN0X3Njb3JlLHRlc3Q9dGVzdCx0cmFpbmluZ19zdGVwcz11cGRhdGVzLGxpbWl0YXRpb25zPXBsYW5bJ2RlY2lzaW9ucyddKSkKICB0bXA9Zm9sZGVyLyd3ZWlnaHRzLnRtcCc7dG9yY2guc2F2ZShiZXN0LHRtcCk7dG1wLnJlcGxhY2UoZm9sZGVyLyd3ZWlnaHRzLnB0JykKICBkLndyaXRlKGZvbGRlci8nRVhQT1JULmpzb24nLGRpY3QocHJvdG9jb2w9cHJvdG9jb2wsam9iPWpvYix3ZWlnaHRzX3NoYTI1Nj1kLnNoYShmb2xkZXIvJ3dlaWdodHMucHQnKSxpZGVudGl0eT1pZGVudGl0eSxpbnB1dF9odz1kLlNQRUNTW2pvYlsnbW9kZWwnXV1bJ2h3J10scHJlcHJvY2Vzc2luZz0nUkdCIGRpcmVjdCByZXNpemU7IFlPTE8gWzAsMV0sIG90aGVycyBJbWFnZU5ldCBub3JtYWxpemF0aW9uJyxjbGFzc2VzPVsnbG93JywnbWlkJywnaGlnaCddIGlmIGpvYlsnbW9kZWwnXSBpbiAoJ21vYmlsZW5ldHY0JywncmVzbmV0NTAnKSBlbHNlIFsndHlyZScsJ3RyZWFkJ10gaWYgam9iWydtb2RlbCddIT0naHJuZXQnIGVsc2UgWydsZWZ0X3VwcGVyJywncmlnaHRfdXBwZXInLCdsZWZ0X21pZGRsZScsJ3JpZ2h0X21pZGRsZScsJ2xlZnRfbG93ZXInLCdyaWdodF9sb3dlciddLHNlbGVjdGVkX2J5PSd2YWxpZGF0aW9uIG9ubHknLHJ1bnRpbWVfc291cmNlcz1jb25maWdbJ3NvdXJjZXMnXSkpCiAgZC53cml0ZShmb2xkZXIvJ0xPQ0FMLmpzb24nLGRpY3Qoc3RhdHVzPSdjb21wbGV0ZWQnLGpvYj1qb2IscHJvdG9jb2w9cHJvdG9jb2wsZXBvY2g9NjAsY3Vyc29yPTAsdXBkYXRlcz11cGRhdGVzKSkKIGV4Y2VwdCBJbnRlcnJ1cHRlZEVycm9yOnJldHVybgogZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZXhjOgogIGQud3JpdGUoZm9sZGVyLydFUlJPUi5qc29uJyxkaWN0KHR5cGU9dHlwZShleGMpLl9fbmFtZV9fLHRyYWNlYmFjaz10cmFjZWJhY2suZm9ybWF0X2V4YygpLHRpbWU9dGltZS50aW1lKCkpKQogIHJhaXNlCgpkZWYgbWFpbigpOgogYT1hcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpO2EuYWRkX2FyZ3VtZW50KCdtb2RlJyxjaG9pY2VzPVsndHJhaW4nLCdzbW9rZScsJ2NvbnRpbnVlLXNtb2tlJ10pO2EuYWRkX2FyZ3VtZW50KCctLWNvbmZpZycscmVxdWlyZWQ9VHJ1ZSk7YS5hZGRfYXJndW1lbnQoJy0tZm9sZGVyJyxyZXF1aXJlZD1UcnVlKTthLmFkZF9hcmd1bWVudCgnLS1kZXZpY2UnLGRlZmF1bHQ9J2N1ZGE6MCcpO2FyZ3M9YS5wYXJzZV9hcmdzKCkKIHNpZ25hbC5zaWduYWwoc2lnbmFsLlNJR0lOVCxzdG9wKTtzaWduYWwuc2lnbmFsKHNpZ25hbC5TSUdURVJNLHN0b3ApCiB0b3JjaC5zZXRfbnVtX3RocmVhZHMoMik7bS5kZXRlcm1pbmlzdGljKCk7Y2ZnPWQucmVhZChhcmdzLmNvbmZpZyk7ZGV2aWNlPXRvcmNoLmRldmljZShhcmdzLmRldmljZSkKIGlmIGFyZ3MubW9kZT09J3RyYWluJzp0cmFpbihjZmcsYXJncy5mb2xkZXIsZGV2aWNlKQogZWxzZTpzbW9rZShjZmcsYXJncy5mb2xkZXIsZGV2aWNlLGFyZ3MubW9kZT09J2NvbnRpbnVlLXNtb2tlJykKaWYgX19uYW1lX189PSdfX21haW5fXyc6bWFpbigpCg=='))
(RUNTIME / 'phase2_hub.py').write_bytes(base64.b64decode('IiIiT25lIHB1Ymxpc2hlciBwZXIgbm90ZWJvb2s7IGltbXV0YWJsZSBzbmFwc2hvdHMsIGZlbmNlZCBsZWFzZXMgYW5kIGJvdW5kZWQgcmV0cmllcy4iIiIKaW1wb3J0IGVtYWlsLnV0aWxzCmltcG9ydCBvcwppbXBvcnQgcmFuZG9tCmltcG9ydCBzaHV0aWwKaW1wb3J0IHRpbWUKaW1wb3J0IHV1aWQKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgZGVxdWUKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsdGltZXpvbmUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gcGhhc2UyX3RyYWluaW5nX2RhdGEgaW1wb3J0IHJlYWQsd3JpdGUsc2hhCgpSRVBPPSdTaGFubXVrNDYyMi90eXJlLXdlYXItc3R1ZHknCmNsYXNzIEh1YjoKIGRlZiBfX2luaXRfXyhzZWxmLHdvcmssd29ya2Vycz0xKToKICBmcm9tIGh1Z2dpbmdmYWNlX2h1YiBpbXBvcnQgSGZBcGkKICBzZWxmLndvcms9UGF0aCh3b3JrKTtzZWxmLmFwaT1IZkFwaSh0b2tlbj1vcy5lbnZpcm9uWydIRl9UT0tFTiddKTtzZWxmLnRva2VuPW9zLmVudmlyb25bJ0hGX1RPS0VOJ10KICBzZWxmLmNhbGxzPWRlcXVlKCk7c2VsZi5saW1pdD1tYXgoOCwxMDAvL3dvcmtlcnMpO3NlbGYub3duZXI9dXVpZC51dWlkNCgpLmhleAogIHNlbGYuYnVkZ2V0X2ZpbGU9c2VsZi53b3JrLydyZXF1ZXN0X2J1ZGdldC5qc29uJwogIGlmIHNlbGYuYnVkZ2V0X2ZpbGUuZXhpc3RzKCk6c2VsZi5jYWxscy5leHRlbmQocmVhZChzZWxmLmJ1ZGdldF9maWxlKSkKIGRlZiByZXRyeShzZWxmLGZuKToKICBmb3IgYXR0ZW1wdCBpbiByYW5nZSg2KToKICAgbm93PXRpbWUudGltZSgpCiAgIHdoaWxlIHNlbGYuY2FsbHMgYW5kIHNlbGYuY2FsbHNbMF08bm93LTM2MDA6c2VsZi5jYWxscy5wb3BsZWZ0KCkKICAgd2hpbGUgbGVuKHNlbGYuY2FsbHMpPj1zZWxmLmxpbWl0OgogICAgdGltZS5zbGVlcChtaW4oMixtYXgoLjAxLHNlbGYuY2FsbHNbMF0rMzYwMC10aW1lLnRpbWUoKSkpKQogICAgd2hpbGUgc2VsZi5jYWxscyBhbmQgc2VsZi5jYWxsc1swXTx0aW1lLnRpbWUoKS0zNjAwOnNlbGYuY2FsbHMucG9wbGVmdCgpCiAgIHNlbGYuY2FsbHMuYXBwZW5kKHRpbWUudGltZSgpKTt3cml0ZShzZWxmLmJ1ZGdldF9maWxlLGxpc3Qoc2VsZi5jYWxscykpCiAgIHRyeTpyZXR1cm4gZm4oKQogICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICByZXNwb25zZT1nZXRhdHRyKGUsJ3Jlc3BvbnNlJyxOb25lKTtzdGF0dXM9Z2V0YXR0cihyZXNwb25zZSwnc3RhdHVzX2NvZGUnLE5vbmUpCiAgICBpZiBzdGF0dXMgbm90IGluIChOb25lLDQyOSw1MDAsNTAyLDUwMyw1MDQpIG9yIGF0dGVtcHQ9PTU6cmFpc2UKICAgIGhpbnQ9Z2V0YXR0cihyZXNwb25zZSwnaGVhZGVycycse30pLmdldCgnUmV0cnktQWZ0ZXInLCcwJykKICAgIHRyeTpkZWxheT1mbG9hdChoaW50KQogICAgZXhjZXB0IChWYWx1ZUVycm9yLFR5cGVFcnJvcik6CiAgICAgdHJ5OmRlbGF5PW1heCgwLChlbWFpbC51dGlscy5wYXJzZWRhdGVfdG9fZGF0ZXRpbWUoaGludCktZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykpLnRvdGFsX3NlY29uZHMoKSkKICAgICBleGNlcHQgRXhjZXB0aW9uOmRlbGF5PTAKICAgIGRlbGF5PW1heChkZWxheSxtaW4oMzAwLDEwKjIqKmF0dGVtcHQpK3JhbmRvbS5SYW5kb20oYXR0ZW1wdCkucmFuZG9tKCkpCiAgICBwcmludChmJ0hGIHRyYW5zaWVudCBIVFRQIHtzdGF0dXN9OyByZXRyeSBpbiB7ZGVsYXk6LjBmfXMuIExvY2FsIGZpbGVzIHJldGFpbmVkLicsZmx1c2g9VHJ1ZSkKICAgIGVuZD10aW1lLm1vbm90b25pYygpK2RlbGF5CiAgICB3aGlsZSB0aW1lLm1vbm90b25pYygpPGVuZDp0aW1lLnNsZWVwKG1pbigyLG1heCguMDEsZW5kLXRpbWUubW9ub3RvbmljKCkpKSkKIGRlZiByZXZpc2lvbihzZWxmKTpyZXR1cm4gc2VsZi5yZXRyeShsYW1iZGE6c2VsZi5hcGkucmVwb19pbmZvKFJFUE8scmVwb190eXBlPSdkYXRhc2V0Jykuc2hhKQogZGVmIGZldGNoKHNlbGYscGF0aCxyZXZpc2lvbixtaXNzaW5nPUZhbHNlKToKICBmcm9tIGh1Z2dpbmdmYWNlX2h1YiBpbXBvcnQgaGZfaHViX2Rvd25sb2FkCiAgZnJvbSBodWdnaW5nZmFjZV9odWIuZXJyb3JzIGltcG9ydCBFbnRyeU5vdEZvdW5kRXJyb3IKICB0cnk6cmV0dXJuIFBhdGgoc2VsZi5yZXRyeShsYW1iZGE6aGZfaHViX2Rvd25sb2FkKFJFUE8scGF0aCxyZXBvX3R5cGU9J2RhdGFzZXQnLHJldmlzaW9uPXJldmlzaW9uLHRva2VuPXNlbGYudG9rZW4sY2FjaGVfZGlyPXN0cihzZWxmLndvcmsvJ2h1Yl9jYWNoZScpKSkpCiAgZXhjZXB0IEVudHJ5Tm90Rm91bmRFcnJvcjoKICAgaWYgbWlzc2luZzpyZXR1cm4gTm9uZQogICByYWlzZQogZGVmIGNsYWltKHNlbGYscHJlZml4LGpvYnMsdGFrZW92ZXI9RmFsc2UpOgogIGZyb20gaHVnZ2luZ2ZhY2VfaHViIGltcG9ydCBDb21taXRPcGVyYXRpb25BZGQKICBzZWxmLnByZWZpeD1wcmVmaXg7c2VsZi5sZWFzZXM9e30KICBmb3IgYXR0ZW1wdCBpbiByYW5nZSg4KToKICAgcmV2PXNlbGYucmV2aXNpb24oKTtwYXRoPXNlbGYuZmV0Y2gocHJlZml4KycvTEVBU0VTLmpzb24nLHJldixUcnVlKTtsZWFzZXM9cmVhZChwYXRoKSBpZiBwYXRoIGVsc2Uge30KICAgZm9yIGpvYiBpbiBqb2JzOgogICAgZW50cnk9bGVhc2VzLmdldChqb2JbJ2lkJ10se30pCiAgICBpZiBlbnRyeS5nZXQoJ293bmVyJykgbm90IGluIChOb25lLHNlbGYub3duZXIpIGFuZCBlbnRyeS5nZXQoJ2V4cGlyZXMnLDApPnRpbWUudGltZSgpIGFuZCBub3QgdGFrZW92ZXI6CiAgICAgcmFpc2UgUnVudGltZUVycm9yKCdSdW4gYWxyZWFkeSBsZWFzZWQ6ICcram9iWydpZCddKycuIFN0b3AgaXRzIG9sZCBub3RlYm9vayBmaXJzdC4gU2V0IFRBS0VfT1ZFUiBvbmx5IGFmdGVyIGNvbmZpcm1pbmcgaXQgc3RvcHBlZC4nKQogICAgbGVhc2VzW2pvYlsnaWQnXV09ZGljdChvd25lcj1zZWxmLm93bmVyLGV4cGlyZXM9dGltZS50aW1lKCkrNTQwMCkKICAgdHJ5OgogICAgc2VsZi5yZXRyeShsYW1iZGE6c2VsZi5hcGkuY3JlYXRlX2NvbW1pdChSRVBPLHJlcG9fdHlwZT0nZGF0YXNldCcscGFyZW50X2NvbW1pdD1yZXYsCiAgICAgb3BlcmF0aW9ucz1bQ29tbWl0T3BlcmF0aW9uQWRkKHBhdGhfaW5fcmVwbz1wcmVmaXgrJy9MRUFTRVMuanNvbicscGF0aF9vcl9maWxlb2JqPV9faW1wb3J0X18oJ2pzb24nKS5kdW1wcyhsZWFzZXMpLmVuY29kZSgpKV0sY29tbWl0X21lc3NhZ2U9J1BoYXNlMiBjbGFpbSAnK3NlbGYub3duZXIpKQogICAgc2VsZi5sZWFzZXM9e2pbJ2lkJ106c2VsZi5vd25lciBmb3IgaiBpbiBqb2JzfTtyZXR1cm4KICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgaWYgZ2V0YXR0cihnZXRhdHRyKGUsJ3Jlc3BvbnNlJyxOb25lKSwnc3RhdHVzX2NvZGUnLE5vbmUpIG5vdCBpbiAoNDA5LDQxMikgb3IgYXR0ZW1wdD09NzpyYWlzZQogZGVmIGNvbW1pdChzZWxmLGZpbGVzLG1lc3NhZ2UscmVsZWFzZT1GYWxzZSk6CiAgZnJvbSBodWdnaW5nZmFjZV9odWIgaW1wb3J0IENvbW1pdE9wZXJhdGlvbkFkZAogICMgZmlsZXMgaXMgcmVtb3RlLXBhdGggLT4gaW1tdXRhYmxlIGxvY2FsIGZpbGUuIFVwbG9hZGluZyBuZXZlciByZWFkcyBsaXZlIHN0YXRlLnB0LgogIGZvciBhdHRlbXB0IGluIHJhbmdlKDgpOgogICByZXY9c2VsZi5yZXZpc2lvbigpO2xwPXNlbGYuZmV0Y2goc2VsZi5wcmVmaXgrJy9MRUFTRVMuanNvbicscmV2LFRydWUpO2xlYXNlcz1yZWFkKGxwKSBpZiBscCBlbHNlIHt9CiAgIGZvciBqb2Isb3duZXIgaW4gc2VsZi5sZWFzZXMuaXRlbXMoKToKICAgIGlmIGxlYXNlcy5nZXQoam9iLHt9KS5nZXQoJ293bmVyJykhPW93bmVyOnJhaXNlIFJ1bnRpbWVFcnJvcignTGVhc2UgZmVuY2luZyByZWplY3RlZCBzdGFsZSB3cml0ZXIgZm9yICcram9iKQogICAgbGVhc2VzW2pvYl09ZGljdChvd25lcj1Ob25lIGlmIHJlbGVhc2UgZWxzZSBvd25lcixleHBpcmVzPTAgaWYgcmVsZWFzZSBlbHNlIHRpbWUudGltZSgpKzU0MDApCiAgIG9wZXJhdGlvbnM9W0NvbW1pdE9wZXJhdGlvbkFkZChwYXRoX2luX3JlcG89ayxwYXRoX29yX2ZpbGVvYmo9c3RyKHYpKSBmb3Igayx2IGluIGZpbGVzLml0ZW1zKCldCiAgIG9wZXJhdGlvbnMuYXBwZW5kKENvbW1pdE9wZXJhdGlvbkFkZChwYXRoX2luX3JlcG89c2VsZi5wcmVmaXgrJy9MRUFTRVMuanNvbicscGF0aF9vcl9maWxlb2JqPV9faW1wb3J0X18oJ2pzb24nKS5kdW1wcyhsZWFzZXMpLmVuY29kZSgpKSkKICAgdHJ5OgogICAgY29tbWl0PXNlbGYucmV0cnkobGFtYmRhOnNlbGYuYXBpLmNyZWF0ZV9jb21taXQoUkVQTyxyZXBvX3R5cGU9J2RhdGFzZXQnLG9wZXJhdGlvbnM9b3BlcmF0aW9ucyxwYXJlbnRfY29tbWl0PXJldixjb21taXRfbWVzc2FnZT1tZXNzYWdlKSkKICAgICMgVmVyaWZ5IHRoZSBjb21taXR0ZWQgc21hbGwgbWFuaWZlc3RzIGF0IHRoZSBleGFjdCBpbW11dGFibGUgcmV2aXNpb24uCiAgICBmb3IgcmVtb3RlLGxvY2FsIGluIGZpbGVzLml0ZW1zKCk6CiAgICAgaWYgcmVtb3RlLmVuZHN3aXRoKCcvTUFOSUZFU1QuanNvbicpOgogICAgICBpZiBzaGEoc2VsZi5mZXRjaChyZW1vdGUsY29tbWl0Lm9pZCkpIT1zaGEobG9jYWwpOnJhaXNlIFJ1bnRpbWVFcnJvcignUmVtb3RlIG1hbmlmZXN0IG1pc21hdGNoJykKICAgIGxhcmdlPXtrOnYgZm9yIGssdiBpbiBmaWxlcy5pdGVtcygpIGlmIHYuc3RhdCgpLnN0X3NpemU+MTAqMTAyNCoqMn0KICAgIGlmIGxhcmdlOgogICAgIGluZm9zPXNlbGYucmV0cnkobGFtYmRhOnNlbGYuYXBpLmdldF9wYXRoc19pbmZvKFJFUE8sbGlzdChsYXJnZSkscmVwb190eXBlPSdkYXRhc2V0JyxyZXZpc2lvbj1jb21taXQub2lkKSkKICAgICBhc3NlcnQge3YucGF0aCBmb3IgdiBpbiBpbmZvc309PXNldChsYXJnZSkKICAgICBmb3IgaW5mbyBpbiBpbmZvczoKICAgICAgaWYgbm90IGluZm8ubGZzIG9yIGluZm8ubGZzLnNoYTI1NiE9c2hhKGxhcmdlW2luZm8ucGF0aF0pOnJhaXNlIFJ1bnRpbWVFcnJvcignUmVtb3RlIGxhcmdlLWZpbGUgU0hBLTI1NiBtaXNtYXRjaCcpCiAgICBwcmlvcj1yZWFkKHNlbGYud29yay8nTEFTVF9SRU1PVEUuanNvbicpLmdldCgnZmlsZXMnLHt9KSBpZiAoc2VsZi53b3JrLydMQVNUX1JFTU9URS5qc29uJykuZXhpc3RzKCkgZWxzZSB7fQogICAgcHJpb3IudXBkYXRlKHtrOnNoYSh2KSBmb3Igayx2IGluIGZpbGVzLml0ZW1zKCl9KQogICAgd3JpdGUoc2VsZi53b3JrLydMQVNUX1JFTU9URS5qc29uJyxkaWN0KHJldmlzaW9uPWNvbW1pdC5vaWQsdGltZT10aW1lLnRpbWUoKSxmaWxlcz1wcmlvcikpCiAgICBwcmludCgnSEYgc25hcHNob3QgY29tbWl0dGVkIGFuZCBtYW5pZmVzdCB2ZXJpZmllZDonLGNvbW1pdC5vaWQsZmx1c2g9VHJ1ZSk7cmV0dXJuIGNvbW1pdC5vaWQKICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgaWYgZ2V0YXR0cihnZXRhdHRyKGUsJ3Jlc3BvbnNlJyxOb25lKSwnc3RhdHVzX2NvZGUnLE5vbmUpIG5vdCBpbiAoNDA5LDQxMikgb3IgYXR0ZW1wdD09NzpyYWlzZQogZGVmIHJlc3RvcmUoc2VsZixqb2IsZm9sZGVyLHJldmlzaW9uKToKICByZW1vdGU9c2VsZi5wcmVmaXgrJy9ydW5zLycram9iWydpZCddO21wPXNlbGYuZmV0Y2gocmVtb3RlKycvTUFOSUZFU1QuanNvbicscmV2aXNpb24sVHJ1ZSkKICBpZiBub3QgbXA6cmV0dXJuIE5vbmUKICBtYW5pZmVzdD1yZWFkKG1wKQogIGlmIG1hbmlmZXN0Wydqb2InXSE9am9iOnJhaXNlIFZhbHVlRXJyb3IoJ1JlbW90ZSBqb2IgaWRlbnRpdHkgbWlzbWF0Y2gnKQogIGZvbGRlcj1QYXRoKGZvbGRlcik7Zm9sZGVyLm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogIGFsbG93ZWQ9eydzdGF0ZS5wdCcsJ3dlaWdodHMucHQnLCdISVNUT1JZLmpzb24nLCdWQUxJREFUSU9OLmpzb24nLCdTTU9LRS5qc29uJywnSURFTlRJVFkuanNvbicsJ01PREVMX0NPTkZJRy5qc29uJywnRVJST1IuanNvbicsJ0ZJTkFMLmpzb24nLCdFWFBPUlQuanNvbicsJ2NvbnNvbGUubG9nJ30KICBpZiBub3Qgc2V0KG1hbmlmZXN0WydmaWxlcyddKTw9YWxsb3dlZDpyYWlzZSBWYWx1ZUVycm9yKCdVbmV4cGVjdGVkIHJlbW90ZSBtYW5pZmVzdCBmaWxlbmFtZScpCiAgaWYgbWFuaWZlc3RbJ3N0YXR1cyddPT0nY29tcGxldGVkJzoKICAgaW5mb3M9c2VsZi5yZXRyeShsYW1iZGE6c2VsZi5hcGkuZ2V0X3BhdGhzX2luZm8oUkVQTyxbcmVtb3RlKycvd2VpZ2h0cy5wdCcscmVtb3RlKycvc3RhdGUucHQnXSxyZXBvX3R5cGU9J2RhdGFzZXQnLHJldmlzaW9uPXJldmlzaW9uKSkKICAgYXNzZXJ0IGxlbihpbmZvcyk9PTIsJ0NvbXBsZXRlZCBydW4gaGFzIG1pc3Npbmcgd2VpZ2h0cy9jaGVja3BvaW50JwogICBmb3IgaW5mbyBpbiBpbmZvczoKICAgIG5hbWU9aW5mby5wYXRoLnJzcGxpdCgnLycsMSlbLTFdCiAgICBpZiBub3QgaW5mby5sZnMgb3IgaW5mby5sZnMuc2hhMjU2IT1tYW5pZmVzdFsnZmlsZXMnXVtuYW1lXTpyYWlzZSBWYWx1ZUVycm9yKCdDb21wbGV0ZWQgY2hlY2twb2ludC93ZWlnaHRzIGludGVncml0eSBtaXNtYXRjaCcpCiAgaWYgKGZvbGRlci8nc3RhdGUucHQnKS5leGlzdHMoKToKICAgZnJvbSBQSUwgaW1wb3J0IEltYWdlCiAgIGltcG9ydCB0b3JjaAogICBsb2NhbD10b3JjaC5sb2FkKGZvbGRlci8nc3RhdGUucHQnLG1hcF9sb2NhdGlvbj0nY3B1Jyx3ZWlnaHRzX29ubHk9RmFsc2UpCiAgIGFzc2VydCBsb2NhbFsnam9iJ109PWpvYiBhbmQgbG9jYWxbJ3Byb3RvY29sJ109PW1hbmlmZXN0Wydwcm90b2NvbCddCiAgIG5ld2VyPWxvY2FsWyd1cGRhdGVzJ10+PW1hbmlmZXN0LmdldCgndXBkYXRlcycsMCkKICAgZGVsIGxvY2FsCiAgIGlmIG5ld2VyIGFuZCBtYW5pZmVzdFsnc3RhdHVzJ10hPSdjb21wbGV0ZWQnOnJldHVybiBtYW5pZmVzdAogICMgQ29tcGxldGVkIGpvYnMgbmVlZCBubyBtb2RlbCBkb3dubG9hZC4gVmVyaWZ5IHNtYWxsIEZJTkFMIG1ldHJpY3MgaW5zdGVhZC4KICB3YW50ZWQ9WydGSU5BTC5qc29uJywnRVhQT1JULmpzb24nXSBpZiBtYW5pZmVzdFsnc3RhdHVzJ109PSdjb21wbGV0ZWQnIGVsc2UgbGlzdChtYW5pZmVzdFsnZmlsZXMnXSkKICBmb3IgbmFtZSBpbiB3YW50ZWQ6CiAgIGV4cGVjdGVkPW1hbmlmZXN0WydmaWxlcyddW25hbWVdO3RhcmdldD1mb2xkZXIvbmFtZQogICBpZiB0YXJnZXQuZXhpc3RzKCkgYW5kIHNoYSh0YXJnZXQpPT1leHBlY3RlZDpjb250aW51ZQogICBkb3dubG9hZGVkPXNlbGYuZmV0Y2gocmVtb3RlKycvJytuYW1lLHJldmlzaW9uKQogICBpZiBzaGEoZG93bmxvYWRlZCkhPWV4cGVjdGVkOnJhaXNlIFZhbHVlRXJyb3IoJ1JlbW90ZSBwYXlsb2FkIGNoZWNrc3VtIG1pc21hdGNoOiAnK25hbWUpCiAgIHNodXRpbC5jb3B5Mihkb3dubG9hZGVkLHRhcmdldCkKICB3cml0ZShmb2xkZXIvJ1JFTU9URS5qc29uJyxtYW5pZmVzdCkKICByZXR1cm4gbWFuaWZlc3QK'))
(RUNTIME / 'phase2_run.py').write_bytes(base64.b64decode('IiIiS2FnZ2xlIGNvb3JkaW5hdG9yOiB0d28gR1BVIHdvcmtlcnMsIG9uZSBIRiB3cml0ZXIsIHN0YXRpYyBjcm9zcy1zZXNzaW9uIHNoYXJkaW5nLiIiIgpmcm9tIFBJTCBpbXBvcnQgSW1hZ2UKaW1wb3J0IGFyZ3BhcnNlLGpzb24sb3Msc2h1dGlsLHNpZ25hbCxzdWJwcm9jZXNzLHN5cyx0aW1lLHRyYWNlYmFjawpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSBmaWxlbG9jayBpbXBvcnQgRmlsZUxvY2sKaW1wb3J0IHRvcmNoCmltcG9ydCBwaGFzZTJfdHJhaW5pbmdfZGF0YSBhcyBkCmZyb20gcGhhc2UyX2h1YiBpbXBvcnQgSHViLFJFUE8KClNUT1A9RmFsc2UKQ0hJTERSRU49W10KZGVmIHN0b3AoKl8pOgogZ2xvYmFsIFNUT1AKIGlmIFNUT1A6cmV0dXJuCiBTVE9QPVRydWUKICMgRm9yd2FyZCBwcm9tcHRseSBldmVuIGlmIHRoZSBjb29yZGluYXRvciBpcyBpbnNpZGUgYW4gSEYgcmV0cnkvYmFja29mZi4KIGZvciBjaGlsZCBpbiBsaXN0KENISUxEUkVOKToKICBpZiBjaGlsZC5wb2xsKCkgaXMgTm9uZTpjaGlsZC5zZW5kX3NpZ25hbChzaWduYWwuU0lHSU5UKQogcHJpbnQoJ1N0b3AgcmVxdWVzdGVkOiB3YWl0aW5nIGZvciBjb21wbGV0ZWQtdXBkYXRlIGNoZWNrcG9pbnRzLCB0aGVuIHB1c2hpbmcgdG8gSEYuJyxmbHVzaD1UcnVlKQoKZGVmIHByZXBhcmVfYXNzZXRzKGh1Yixmb2xkZXIsbW9kZWxzKToKIGZyb20gaHVnZ2luZ2ZhY2VfaHViIGltcG9ydCBoZl9odWJfZG93bmxvYWQKIGltcG9ydCByZXF1ZXN0cwogZm9sZGVyPVBhdGgoZm9sZGVyKTtpZGVudGl0aWVzPXt9CiBmb3IgbmFtZSBpbiBtb2RlbHM6CiAgc3BlYz1kLlNQRUNTW25hbWVdO291dD1mb2xkZXIvbmFtZTtvdXQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgaWYgbmFtZT09J3lvbG8yNm0nOgogICB0YXJnZXQ9b3V0Lyd5b2xvMjZtLXNlZy5wdCcKICAgaWYgbm90IHRhcmdldC5leGlzdHMoKToKICAgIGRlZiBkb3dubG9hZCgpOgogICAgIHdpdGggcmVxdWVzdHMuZ2V0KHNwZWNbJ3VybCddLHN0cmVhbT1UcnVlLHRpbWVvdXQ9NjApIGFzIHJlc3BvbnNlOgogICAgICByZXNwb25zZS5yYWlzZV9mb3Jfc3RhdHVzKCkKICAgICAgd2l0aCBvcGVuKHRhcmdldC53aXRoX3N1ZmZpeCgnLnBhcnQnKSwnd2InKSBhcyBmOgogICAgICAgZm9yIGNodW5rIGluIHJlc3BvbnNlLml0ZXJfY29udGVudCgxMDI0KioyKTpmLndyaXRlKGNodW5rKQogICAgIHRhcmdldC53aXRoX3N1ZmZpeCgnLnBhcnQnKS5yZXBsYWNlKHRhcmdldCkKICAgIGh1Yi5yZXRyeShkb3dubG9hZCkKICAgaWYgZC5zaGEodGFyZ2V0KSE9c3BlY1snc2hhMjU2J106cmFpc2UgVmFsdWVFcnJvcignT2ZmaWNpYWwgWU9MTyB3ZWlnaHRzIGhhc2ggbWlzbWF0Y2gnKQogIGVsc2U6CiAgIG5hbWVzPVtzcGVjWydmaWxlJ11dKyhbJ2NvbmZpZy5qc29uJ10gaWYgbmFtZT09J3NlZ2Zvcm1lcicgZWxzZSBbXSkKICAgZm9yIGZuIGluIG5hbWVzOgogICAgdGFyZ2V0PW91dC9mbgogICAgaWYgbm90IHRhcmdldC5leGlzdHMoKToKICAgICBzcmM9aHViLnJldHJ5KGxhbWJkYTpoZl9odWJfZG93bmxvYWQoc3BlY1sncmVwbyddLGZuLHJldmlzaW9uPXNwZWNbJ3JldmlzaW9uJ10sdG9rZW49aHViLnRva2VuLGNhY2hlX2Rpcj1zdHIoZm9sZGVyLydjYWNoZScpKSkKICAgICBzaHV0aWwuY29weTIoc3JjLHRhcmdldCkKICBpZGVudGl0aWVzW25hbWVdPXtwLm5hbWU6ZC5zaGEocCkgZm9yIHAgaW4gb3V0Lml0ZXJkaXIoKSBpZiBwLmlzX2ZpbGUoKX0KIHJldHVybiBpZGVudGl0aWVzCgpkZWYgc25hcHNob3QoaHViLHdvcmsscnVucm9vdCxzZWxlY3RlZCxwcm90b2NvbCxyZWxlYXNlPUZhbHNlKToKIHN0YWdpbmc9d29yay8nc25hcHNob3QnO3N0YWdpbmcubWtkaXIoZXhpc3Rfb2s9VHJ1ZSk7ZmlsZXM9e30KIG5lZWRlZD1zdW0ocC5zdGF0KCkuc3Rfc2l6ZSBmb3IgaiBpbiBzZWxlY3RlZCBmb3IgcCBpbiAocnVucm9vdC9qWydpZCddKS5nbG9iKCcqLnB0JykpCiByZXVzYWJsZT1zdW0ocC5zdGF0KCkuc3Rfc2l6ZSBmb3IgcCBpbiBzdGFnaW5nLnJnbG9iKCcqLnB0JykpCiBvdXRwdXRfYnl0ZXM9c3VtKHAuc3RhdCgpLnN0X3NpemUgZm9yIHAgaW4gd29yay5yZ2xvYignKicpIGlmIHAuaXNfZmlsZSgpKQogaWYgb3V0cHV0X2J5dGVzK25lZWRlZC1yZXVzYWJsZT4xOCoxMDI0KiozOgogIHJhaXNlIFJ1bnRpbWVFcnJvcignU25hcHNob3Qgd291bGQgZXhjZWVkIHRoZSAxOCBHaUIgd29ya2luZy1vdXRwdXQgYnVkZ2V0OyBjaGVja3BvaW50cyByZXRhaW5lZCBpbiBzY3JhdGNoLiBSZXN1bWUgaW4gYSBmcmVzaCBzZXNzaW9uIGZyb20gdGhlIGxhc3QgdmVyaWZpZWQgSEYgc25hcHNob3QuJykKIGlmIHNodXRpbC5kaXNrX3VzYWdlKHdvcmspLmZyZWUrcmV1c2FibGU8bmVlZGVkKzEwMjQqKjM6CiAgcmFpc2UgUnVudGltZUVycm9yKCdJbnN1ZmZpY2llbnQgZnJlZSBzcGFjZSBmb3IgYW4gaW1tdXRhYmxlIEhGIHNuYXBzaG90OyBsaXZlIGNoZWNrcG9pbnRzIHJldGFpbmVkIGluICcrc3RyKHJ1bnJvb3QpKQogIyBFYWNoIGltbXV0YWJsZSBzbmFwc2hvdCBpcyByZXBsYWNlZCBvbmx5IGFmdGVyIHRoZSBwcmVjZWRpbmcgdXBsb2FkIHJldHVybmVkLgogZm9yIGpvYiBpbiBzZWxlY3RlZDoKICBmb2xkZXI9cnVucm9vdC9qb2JbJ2lkJ107ZGVzdD1zdGFnaW5nL2pvYlsnaWQnXTtkZXN0Lm1rZGlyKGV4aXN0X29rPVRydWUpCiAgaWYgKGZvbGRlci8nUkVNT1RFLmpzb24nKS5leGlzdHMoKSBhbmQgZC5yZWFkKGZvbGRlci8nUkVNT1RFLmpzb24nKVsnc3RhdHVzJ109PSdjb21wbGV0ZWQnOmNvbnRpbnVlCiAgc3RhdHVzPWZvbGRlci8nTE9DQUwuanNvbic7Y2hlY2twb2ludD1mb2xkZXIvJ3N0YXRlLnB0JwogIG5hbWVzPVtdO3N1bW1hcnk9ZGljdChzdGF0dXM9J25vdF9zdGFydGVkJyx1cGRhdGVzPTAsam9iPWpvYixwcm90b2NvbD1wcm90b2NvbCkKICBpZiBjaGVja3BvaW50LmV4aXN0cygpOgogICB3aXRoIEZpbGVMb2NrKHN0cihmb2xkZXIvJ2NoZWNrcG9pbnQubG9jaycpKToKICAgIHNodXRpbC5jb3B5MihjaGVja3BvaW50LGRlc3QvJ3N0YXRlLnB0JykKICAgIGxvY2FsPWQucmVhZChzdGF0dXMpIGlmIHN0YXR1cy5leGlzdHMoKSBlbHNlIHt9CiAgIHNhdmVkPXRvcmNoLmxvYWQoZGVzdC8nc3RhdGUucHQnLG1hcF9sb2NhdGlvbj0nY3B1Jyx3ZWlnaHRzX29ubHk9RmFsc2UpCiAgIGFzc2VydCBzYXZlZFsncHJvdG9jb2wnXT09cHJvdG9jb2wgYW5kIHNhdmVkWydqb2InXT09am9iCiAgIHN1bW1hcnkudXBkYXRlKGVwb2NoPXNhdmVkWydlcG9jaCddLGN1cnNvcj1zYXZlZFsnY3Vyc29yJ10sdXBkYXRlcz1zYXZlZFsndXBkYXRlcyddLHN0YXR1cz0nY29tcGxldGVkJyBpZiBsb2NhbC5nZXQoJ3N0YXR1cycpPT0nY29tcGxldGVkJyBhbmQgc2F2ZWRbJ2Vwb2NoJ109PTYwIGVsc2UgJ3Jlc3VtYWJsZScpCiAgIGQud3JpdGUoZGVzdC8nSElTVE9SWS5qc29uJyxzYXZlZFsnaGlzdG9yeSddKTtkLndyaXRlKGRlc3QvJ1ZBTElEQVRJT04uanNvbicsc2F2ZWRbJ3ZhbGlkYXRpb24nXSk7ZGVsIHNhdmVkCiAgIG5hbWVzKz1bJ3N0YXRlLnB0JywnSElTVE9SWS5qc29uJywnVkFMSURBVElPTi5qc29uJ10KICBlbGlmIChmb2xkZXIvJ1JFTU9URS5qc29uJykuZXhpc3RzKCkgYW5kIGQucmVhZChmb2xkZXIvJ1JFTU9URS5qc29uJylbJ3N0YXR1cyddPT0nY29tcGxldGVkJzpjb250aW51ZQogIGZvciBuYW1lIGluICgnU01PS0UuanNvbicsJ0lERU5USVRZLmpzb24nLCdNT0RFTF9DT05GSUcuanNvbicsJ0VSUk9SLmpzb24nLCdGSU5BTC5qc29uJywnRVhQT1JULmpzb24nLCd3ZWlnaHRzLnB0Jyk6CiAgIHNvdXJjZT1mb2xkZXIvbmFtZQogICBpZiBzb3VyY2UuZXhpc3RzKCk6c2h1dGlsLmNvcHkyKHNvdXJjZSxkZXN0L25hbWUpO25hbWVzLmFwcGVuZChuYW1lKQogICMgTG9ncyBhcmUgc3VwcGxlbWVudGFyeTsgY2hlY2twb2ludCBoaXN0b3J5IGlzIGF1dGhvcml0YXRpdmUgYW5kIGF0b21pYy4KICBpZiAoZm9sZGVyLydjb25zb2xlLmxvZycpLmV4aXN0cygpOnNodXRpbC5jb3B5Mihmb2xkZXIvJ2NvbnNvbGUubG9nJyxkZXN0Lydjb25zb2xlLmxvZycpO25hbWVzLmFwcGVuZCgnY29uc29sZS5sb2cnKQogIGlmIG5vdCBuYW1lczpjb250aW51ZQogIHN1bW1hcnlbJ2ZpbGVzJ109e25hbWU6ZC5zaGEoZGVzdC9uYW1lKSBmb3IgbmFtZSBpbiBuYW1lc307ZC53cml0ZShkZXN0LydNQU5JRkVTVC5qc29uJyxzdW1tYXJ5KQogIGZvciBuYW1lIGluIG5hbWVzK1snTUFOSUZFU1QuanNvbiddOmZpbGVzW2h1Yi5wcmVmaXgrJy9ydW5zLycram9iWydpZCddKycvJytuYW1lXT1kZXN0L25hbWUKIGZvciBuYW1lIGluICgnQ09OVFJBQ1QuanNvbicsJ09WRVJMQVkuanNvbicsJ0VOVklST05NRU5ULmpzb24nLCdTRVNTSU9OLmpzb24nLCdBU1NFVFMuanNvbicsJ1NFU1NJT05fRVJST1IuanNvbicpOgogIGlmICh3b3JrL25hbWUpLmV4aXN0cygpOmZpbGVzW2h1Yi5wcmVmaXgrJy9zZXNzaW9ucy8nK2h1Yi5vd25lcisnLycrbmFtZV09d29yay9uYW1lCiBmb3Igc291cmNlIGluIHNvcnRlZChQYXRoKF9fZmlsZV9fKS5wYXJlbnQuZ2xvYigncGhhc2UyXyoucHknKSk6CiAgIyBFeHBsaWNpdCBlbWJlZGRlZCBydW50aW1lIGRpcmVjdG9yeSwgbmV2ZXIgYXJiaXRyYXJ5IHByb2plY3Qgb3Igc2VjcmV0IGZpbGVzLgogIGZpbGVzW2h1Yi5wcmVmaXgrJy9zb3VyY2VzLycrc291cmNlLm5hbWVdPXNvdXJjZQogIyBSZW1vdmUgdW5jaGFuZ2VkIHBheWxvYWRzIGZyb20gcmVwZWF0IGNvbW1pdHMgd2hpbGUgcmV0YWluaW5nIG1hbmlmZXN0cy4KIHByaW9yPWQucmVhZCh3b3JrLydMQVNUX1JFTU9URS5qc29uJykuZ2V0KCdmaWxlcycse30pIGlmICh3b3JrLydMQVNUX1JFTU9URS5qc29uJykuZXhpc3RzKCkgZWxzZSB7fQogZmlsZXM9e2s6diBmb3Igayx2IGluIGZpbGVzLml0ZW1zKCkgaWYgcHJpb3IuZ2V0KGspIT1kLnNoYSh2KSBvciBrLmVuZHN3aXRoKCcvTUFOSUZFU1QuanNvbicpfQogcmV2aXNpb249aHViLmNvbW1pdChmaWxlcywnUGhhc2UyIGNoZWNrcG9pbnQgYmF0Y2gnKygnIGFuZCByZWxlYXNlJyBpZiByZWxlYXNlIGVsc2UgJycpLHJlbGVhc2U9cmVsZWFzZSkKICMgQ29tbWl0IGlzIHRyYW5zYWN0aW9uYWw7IGZpbGUgaGFzaGVzIGFyZSByZWNvcmRlZCBpbiByZXZpc2lvbi1waW5uZWQgbWFuaWZlc3RzLgogIyBLZWVwIGxvY2FsIHBlbmRpbmcgc3RhdGUgaWYgYW55IGNhbGwgYWJvdmUgZmFpbHMuIENsZWFuIG9ubHkgb3VyIHN0YWdpbmcgdHJlZS4KIGZvciBjaGlsZCBpbiBzdGFnaW5nLml0ZXJkaXIoKToKICBhc3NlcnQgY2hpbGQucmVzb2x2ZSgpLnBhcmVudD09c3RhZ2luZy5yZXNvbHZlKCkKICBzaHV0aWwucm10cmVlKGNoaWxkKQogcmV0dXJuIHJldmlzaW9uCgpkZWYgcnVuKGFyZ3MpOgogd29yaz1QYXRoKGFyZ3Mud29yaykucmVzb2x2ZSgpO3dvcmsubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiByb290PWQubG9jYXRlKGFyZ3Mucm9vdCkKIGZyb20gcGhhc2UyX2RhdGFzZXRfdmVyaWZ5IGltcG9ydCB2ZXJpZnkKIHZlcmlmeShyb290KQogcGxhbj1kLm92ZXJsYXkocm9vdCk7bW9kZWxzPWFyZ3MubW9kZWxzLnNwbGl0KCcsJyk7Y29uZGl0aW9ucz1hcmdzLmNvbmRpdGlvbnMuc3BsaXQoJywnKQogYXNzZXJ0IHNldChtb2RlbHMpPD1zZXQoZC5NT0RFTFMpIGFuZCBzZXQoY29uZGl0aW9ucyk8PXsnY29tYmluZWQnLCdvbGRfb25seSd9CiBhc3NlcnQgMDw9YXJncy53b3JrZXI8YXJncy53b3JrZXJzPD00CiAjIFN0YWJsZSBmdWxsIHN0dWR5IG9yZGVyIG1lYW5zIGRpZmZlcmVudCBtb2RlbCBub3RlYm9va3MgdXNlIHRoZSBzYW1lIG93bmVyc2hpcC4KIGFsbGpvYnM9ZC5qb2JzKCgnY29tYmluZWQnLCdvbGRfb25seScpKTtzZWxlY3RlZD1baiBmb3IgaSxqIGluIGVudW1lcmF0ZShhbGxqb2JzKSBpZiBqWydtb2RlbCddIGluIG1vZGVscyBhbmQgalsnY29uZGl0aW9uJ10gaW4gY29uZGl0aW9ucyBhbmQgaSVhcmdzLndvcmtlcnM9PWFyZ3Mud29ya2VyXQogaWYgbm90IHNlbGVjdGVkOnByaW50KCdObyBqb2JzIGFzc2lnbmVkIHRvIHRoaXMgd29ya2VyLicpO3JldHVybgogYXNzZXJ0IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCksJ1NlbGVjdCBLYWdnbGUgR1BVIFQ0IHgyJwogc2xvdHM9bWluKDIsdG9yY2guY3VkYS5kZXZpY2VfY291bnQoKSk7YXNzZXJ0IHNsb3RzPj0xCiBpbXBvcnQgcHN1dGlsCiBpZiBwc3V0aWwudmlydHVhbF9tZW1vcnkoKS5hdmFpbGFibGU8NCoxMDI0KiozOnJhaXNlIFJ1bnRpbWVFcnJvcignTmVlZCBhdCBsZWFzdCA0IEdpQiBhdmFpbGFibGUgaG9zdCBSQU0gYmVmb3JlIHN0YXJ0aW5nIEdQVSB3b3JrZXJzJykKIHNvdXJjZXM9e3AubmFtZTpkLnNoYShwKSBmb3IgcCBpbiBQYXRoKF9fZmlsZV9fKS5wYXJlbnQuZ2xvYigncGhhc2UyXyoucHknKX0KIGltcG9ydCBpbXBvcnRsaWIubWV0YWRhdGEKIHZlcnNpb25zPXt4OmltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKHgpIGZvciB4IGluIFsndG9yY2gnLCd0b3JjaHZpc2lvbicsJ3RpbW0nLCd0cmFuc2Zvcm1lcnMnLCd1bHRyYWx5dGljcycsJ2h1Z2dpbmdmYWNlX2h1YicsJ251bXB5JywnUGlsbG93Jywnc2FmZXRlbnNvcnMnLCdmaWxlbG9jayddfQogY29udHJhY3Q9ZGljdCh2ZXJzaW9uPWQuUkVWSVNJT04sc291cmNlX2NoZWNrc3VtPWQuUkVMRUFTRV9TSEEsb3ZlcmxheV9zaGEyNTY9ZC5kaWdlc3QocGxhbiksbW9kZWxzPWQuU1BFQ1Msc291cmNlcz1zb3VyY2VzLHBhY2thZ2VzPXtrOnYgZm9yIGssdiBpbiB2ZXJzaW9ucy5pdGVtcygpIGlmIGsgbm90IGluICgndG9yY2gnLCd0b3JjaHZpc2lvbicsJ251bXB5JywnUGlsbG93Jyl9LAogIGVwb2Nocz02MCxzZWVkcz1bMSwyLDNdLGxyPS4wMDAxLHdlaWdodF9kZWNheT0uMDEscHJlY2lzaW9uPSdBTVAgRlAxNiB3aXRoIHNhbWUtYmF0Y2ggYm91bmRlZCByZXRyaWVzIGFuZCBsb2dnZWQgRlAzMiBmaW5hbCBhdHRlbXB0JywKICBzYW1wbGVyPSdjbGFzcyB0aGVuIGRvbWFpbiB0aGVuIG9yaWdpbmFsLXR5cmUvdmlkZW8tY2xpcCBiYWxhbmNlZDsgc3RhdGVsZXNzIHJlcGxhY2VtZW50OyBlcXVhbCBvbGQvY29tYmluZWQgdXBkYXRlIGJ1ZGdldHMnLAogIGF1Z21lbnRhdGlvbj0nc3RhdGVsZXNzIGhvcml6b250YWwgZmxpcCB3aXRoIHBvaW50IHN3YXA7IGJyaWdodG5lc3MvY29udHJhc3Q7IG9jY2FzaW9uYWwgbWlsZCBibHVyJywKICBjaGVja3BvaW50PSdldmVyeSBjb21wbGV0ZWQgb3B0aW1pemVyIHVwZGF0ZTsgY3Vyc29yLCBSTkcsIG9wdGltaXplciwgc2NhbGVyLCBzY2hlZHVsZXIsIGhpc3RvcnksIHZhbGlkYXRpb24gYW5kIGJlc3Qgd2VpZ2h0cycsCiAgc2VsZWN0aW9uPSd2YWxpZGF0aW9uIGJlc3QsIGVhcmxpZXN0IHRpZTsgdGVzdCBvbmx5IGF0IGNvbXBsZXRpb24nLGRlY2lzaW9ucz1wbGFuWydkZWNpc2lvbnMnXSkKIHByb3RvY29sPWQuZGlnZXN0KGNvbnRyYWN0KTtwcmVmaXg9J3BoYXNlMi90cmFpbmluZy1yMS8nK3Byb3RvY29sCiBwcmludCgnUlVOX1BSRUZJWCA9JyxwcmVmaXgsZmx1c2g9VHJ1ZSkKIHByaW50KCdUcmFpbmluZyBvdmVybGF5OicscGxhblsnY291bnRzJ10sJ0dQVSBwcm9jZXNzZXM6JyxzbG90cywnYXNzaWduZWQgam9iczonLGxlbihzZWxlY3RlZCksZmx1c2g9VHJ1ZSkKIGQud3JpdGUod29yay8nQ09OVFJBQ1QuanNvbicsY29udHJhY3QpO2Qud3JpdGUod29yay8nT1ZFUkxBWS5qc29uJyxwbGFuKQogZC53cml0ZSh3b3JrLydFTlZJUk9OTUVOVC5qc29uJyxkaWN0KHBhY2thZ2VzPXZlcnNpb25zLHB5dGhvbj1zeXMudmVyc2lvbixncHVzPVt0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZShpKSBmb3IgaSBpbiByYW5nZShzbG90cyldLGdwdV9ieXRlcz1bdG9yY2guY3VkYS5nZXRfZGV2aWNlX3Byb3BlcnRpZXMoaSkudG90YWxfbWVtb3J5IGZvciBpIGluIHJhbmdlKHNsb3RzKV0saG9zdF9yYW1fYXZhaWxhYmxlPXBzdXRpbC52aXJ0dWFsX21lbW9yeSgpLmF2YWlsYWJsZSxvdXRwdXRfZnJlZT1zaHV0aWwuZGlza191c2FnZSh3b3JrKS5mcmVlLHNjcmF0Y2hfZnJlZT1zaHV0aWwuZGlza191c2FnZSgnL3RtcCcpLmZyZWUpKQogZC53cml0ZSh3b3JrLydTRVNTSU9OLmpzb24nLGRpY3Qod29ya2VyPWFyZ3Mud29ya2VyLHdvcmtlcnM9YXJncy53b3JrZXJzLHNlbGVjdGVkPXNlbGVjdGVkLHNsb3RzPXNsb3RzLG1vZGU9YXJncy5tb2RlLGNvbmRpdGlvbnM9Y29uZGl0aW9ucykpCiBodWI9SHViKHdvcmssYXJncy53b3JrZXJzKTtodWIuY2xhaW0ocHJlZml4LHNlbGVjdGVkLGFyZ3MudGFrZV9vdmVyKQogcnVucm9vdD1QYXRoKCcvdG1wJykvKCdwaGFzZTJfJytwcm90b2NvbCkvKCd3b3JrZXInK3N0cihhcmdzLndvcmtlcikpLydydW5zJztydW5yb290Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogYXNzZXRzPXdvcmsvJ2Fzc2V0cyc7YWN0aXZlPXt9O2xvZ3M9e307cmVtYWluaW5nPVtdO3N0YXJ0PXRpbWUubW9ub3RvbmljKCk7bGFzdF9wdXNoPXN0YXJ0CiB0cnk6CiAgYXNzZXRfaWRzPXByZXBhcmVfYXNzZXRzKGh1Yixhc3NldHMsbW9kZWxzKTtkLndyaXRlKHdvcmsvJ0FTU0VUUy5qc29uJyxhc3NldF9pZHMpCiAgcmV2PWh1Yi5yZXZpc2lvbigpCiAgZm9yIGpvYiBpbiBzZWxlY3RlZDoKICAgZm9sZGVyPXJ1bnJvb3Qvam9iWydpZCddO2ZvbGRlci5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgcmVtb3RlPWh1Yi5yZXN0b3JlKGpvYixmb2xkZXIscmV2KQogICBpZiByZW1vdGUgYW5kIHJlbW90ZVsnc3RhdHVzJ109PSdjb21wbGV0ZWQnIGFuZCBhcmdzLm1vZGU9PSd0cmFpbic6cHJpbnQoJ1ZlcmlmaWVkIGNvbXBsZXRlZDsgc2tpcCcsam9iWydpZCddLGZsdXNoPVRydWUpO2NvbnRpbnVlCiAgIHJlbWFpbmluZy5hcHBlbmQoam9iKQogIGlmIGFyZ3MubW9kZT09J3Ntb2tlJzoKICAgIyBPbmUgcmVhbC1tb2RlbCBzZWFtIHBlciBzZWxlY3RlZCBhcmNoaXRlY3R1cmUsIG5vdCBldmVyeSBzZWVkL2NvbmRpdGlvbi4KICAgc2Vlbj1zZXQoKTtyZW1haW5pbmc9W2ogZm9yIGogaW4gcmVtYWluaW5nIGlmIG5vdCAoalsnbW9kZWwnXSBpbiBzZWVuIG9yIHNlZW4uYWRkKGpbJ21vZGVsJ10pKV0KICB3aGlsZSByZW1haW5pbmcgb3IgYWN0aXZlOgogICBpZiB0aW1lLm1vbm90b25pYygpLXN0YXJ0PjguNSozNjAwOnN0b3AoKQogICBmb3Igc2xvdCBpbiByYW5nZShzbG90cyk6CiAgICBpZiBTVE9QIG9yIHNsb3QgaW4gYWN0aXZlIG9yIG5vdCByZW1haW5pbmc6Y29udGludWUKICAgIGpvYj1yZW1haW5pbmcucG9wKDApO2ZvbGRlcj1ydW5yb290L2pvYlsnaWQnXTtjZmc9ZGljdChwbGFuPXBsYW4sam9iPWpvYixwcm90b2NvbD1wcm90b2NvbCxyb290PXN0cihyb290KSxhc3NldHM9c3RyKGFzc2V0cyksc291cmNlcz1zb3VyY2VzLGFzc2V0X2lkcz1hc3NldF9pZHMpCiAgICBjb25maWc9Zm9sZGVyLydjb25maWcuanNvbic7ZC53cml0ZShjb25maWcsY2ZnKQogICAgZW52PW9zLmVudmlyb24uY29weSgpO2Vudi5wb3AoJ0hGX1RPS0VOJyxOb25lKTtlbnZbJ0NVREFfVklTSUJMRV9ERVZJQ0VTJ109c3RyKHNsb3QpCiAgICBlbnZbJ0hGX0hVQl9PRkZMSU5FJ109JzEnO2VudlsnVFJBTlNGT1JNRVJTX09GRkxJTkUnXT0nMSc7ZW52WydDVUJMQVNfV09SS1NQQUNFX0NPTkZJRyddPSc6NDA5Njo4JztlbnZbJ09NUF9OVU1fVEhSRUFEUyddPScyJztlbnZbJ1BZVEhPTkRPTlRXUklURUJZVEVDT0RFJ109JzEnCiAgICBlbnZbJ1lPTE9fQ09ORklHX0RJUiddPXN0cih3b3JrLyd5b2xvX3NldHRpbmdzJykKICAgICh3b3JrLyd5b2xvX3NldHRpbmdzJy8nVWx0cmFseXRpY3MnKS5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgIGxvZz1vcGVuKGZvbGRlci8nY29uc29sZS5sb2cnLCdhJyxlbmNvZGluZz0ndXRmLTgnKTtsb2dzW3Nsb3RdPWxvZwogICAgcHJvY2Vzcz1zdWJwcm9jZXNzLlBvcGVuKFtzeXMuZXhlY3V0YWJsZSwnLUInLCctdScsc3RyKFBhdGgoX19maWxlX18pLndpdGhfbmFtZSgncGhhc2UyX3dvcmtlci5weScpKSxhcmdzLm1vZGUsJy0tY29uZmlnJyxzdHIoY29uZmlnKSwnLS1mb2xkZXInLHN0cihmb2xkZXIpLCctLWRldmljZScsJ2N1ZGE6MCddLGVudj1lbnYsc3Rkb3V0PWxvZyxzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQpCiAgICBDSElMRFJFTi5hcHBlbmQocHJvY2VzcykKICAgIGFjdGl2ZVtzbG90XT0ocHJvY2Vzcyxqb2IsRmFsc2UpO3ByaW50KGYiR1BVIHtzbG90fToge2pvYlsnaWQnXX0gc3RhcnRlZCAocmVzdW1lIGNoZWNrIGZpcnN0KSIsZmx1c2g9VHJ1ZSkKICAgZmluaXNoZWQ9RmFsc2UKICAgZm9yIHNsb3QsKHByb2Nlc3Msam9iLHNpZ25hbGVkKSBpbiBsaXN0KGFjdGl2ZS5pdGVtcygpKToKICAgIGlmIFNUT1AgYW5kIG5vdCBzaWduYWxlZDoKICAgICBwcm9jZXNzLnNlbmRfc2lnbmFsKHNpZ25hbC5TSUdJTlQpO2FjdGl2ZVtzbG90XT0ocHJvY2Vzcyxqb2IsVHJ1ZSkKICAgIGNvZGU9cHJvY2Vzcy5wb2xsKCkKICAgIGlmIGNvZGUgaXMgbm90IE5vbmU6CiAgICAgbG9ncy5wb3Aoc2xvdCkuY2xvc2UoKTtkZWwgYWN0aXZlW3Nsb3RdO2ZpbmlzaGVkPVRydWUKICAgICBDSElMRFJFTi5yZW1vdmUocHJvY2VzcykKICAgICB0YWlsPShydW5yb290L2pvYlsnaWQnXS8nY29uc29sZS5sb2cnKS5yZWFkX3RleHQoZW5jb2Rpbmc9J3V0Zi04JyxlcnJvcnM9J3JlcGxhY2UnKS5zcGxpdGxpbmVzKClbLTEyOl0KICAgICBwcmludCgnXG4nLmpvaW4odGFpbCksZmx1c2g9VHJ1ZSkKICAgICBpZiBjb2RlIGFuZCBub3QgU1RPUDpyYWlzZSBSdW50aW1lRXJyb3IoZiJ7am9iWydpZCddfSBmYWlsZWQ7IHNlZSByZXRhaW5lZCBjb25zb2xlLmxvZyIpCiAgICAgcHJpbnQoam9iWydpZCddLCd3b3JrZXIgc3RvcHBlZDsgcHVibGlzaGluZyBwcm9ncmVzcycsZmx1c2g9VHJ1ZSkKICAgaWYgZmluaXNoZWQgb3IgdGltZS5tb25vdG9uaWMoKS1sYXN0X3B1c2g+PTE4MDA6CiAgICBzbmFwc2hvdChodWIsd29yayxydW5yb290LHNlbGVjdGVkLHByb3RvY29sKTtsYXN0X3B1c2g9dGltZS5tb25vdG9uaWMoKQogICBpZiBTVE9QIGFuZCBub3QgYWN0aXZlOmJyZWFrCiAgICMgU21hbGwgb3V0cHV0IHN0YXR1cyBldmVyeSBtaW51dGU7IHRyYWluaW5nIGxvZ3MgcmV0YWluIGRldGFpbGVkIGVwb2NoIGxpbmVzLgogICBpZiBpbnQodGltZS5tb25vdG9uaWMoKS1zdGFydCklNjA8MjoKICAgIGZvciBfLGpvYixfIGluIGFjdGl2ZS52YWx1ZXMoKToKICAgICBzdGF0dXM9cnVucm9vdC9qb2JbJ2lkJ10vJ0xPQ0FMLmpzb24nCiAgICAgcHJpbnQoam9iWydpZCddLGQucmVhZChzdGF0dXMpIGlmIHN0YXR1cy5leGlzdHMoKSBlbHNlICdtb2RlbC9yZXN1bWUgcHJlZmxpZ2h0IHJ1bm5pbmcnLGZsdXNoPVRydWUpCiAgIHRpbWUuc2xlZXAoMikKIGV4Y2VwdCBCYXNlRXhjZXB0aW9uOgogIGQud3JpdGUod29yay8nU0VTU0lPTl9FUlJPUi5qc29uJyxkaWN0KHRyYWNlYmFjaz10cmFjZWJhY2suZm9ybWF0X2V4YygpLHRpbWU9dGltZS50aW1lKCkpKQogIHJhaXNlCiBmaW5hbGx5OgogICMgQ2F0Y2hhYmxlIGVycm9ycyBzdG9wIGFsbCBjaGlsZCB3cml0ZXJzIGJlZm9yZSB0aGUgZW1lcmdlbmN5IHNuYXBzaG90LgogIGZvciBwcm9jZXNzLF8sXyBpbiBhY3RpdmUudmFsdWVzKCk6CiAgIGlmIHByb2Nlc3MucG9sbCgpIGlzIE5vbmU6cHJvY2Vzcy5zZW5kX3NpZ25hbChzaWduYWwuU0lHSU5UKQogIGZvciBwcm9jZXNzLF8sXyBpbiBhY3RpdmUudmFsdWVzKCk6cHJvY2Vzcy53YWl0KCkKICBmb3IgbG9nIGluIGxvZ3MudmFsdWVzKCk6bG9nLmNsb3NlKCkKICBzbmFwc2hvdChodWIsd29yayxydW5yb290LHNlbGVjdGVkLHByb3RvY29sLHJlbGVhc2U9VHJ1ZSkKIHByaW50KCdTZXNzaW9uIGZpbmlzaGVkLiBSZXJ1biB1bmNoYW5nZWQgbm90ZWJvb2sgZm9yIHBlbmRpbmcgam9iczsgdmVyaWZpZWQgY29tcGxldGVkIGpvYnMgYXJlIHNraXBwZWQuJyxmbHVzaD1UcnVlKQoKaWYgX19uYW1lX189PSdfX21haW5fXyc6CiBwPWFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCk7cC5hZGRfYXJndW1lbnQoJ21vZGUnLGNob2ljZXM9Wyd0cmFpbicsJ3Ntb2tlJ10pO3AuYWRkX2FyZ3VtZW50KCctLW1vZGVscycsZGVmYXVsdD0nLCcuam9pbihkLk1PREVMUykpO3AuYWRkX2FyZ3VtZW50KCctLWNvbmRpdGlvbnMnLGRlZmF1bHQ9J2NvbWJpbmVkJyk7cC5hZGRfYXJndW1lbnQoJy0td29ya2VyJyx0eXBlPWludCxkZWZhdWx0PTApO3AuYWRkX2FyZ3VtZW50KCctLXdvcmtlcnMnLHR5cGU9aW50LGRlZmF1bHQ9MSk7cC5hZGRfYXJndW1lbnQoJy0tcm9vdCcsZGVmYXVsdD0nJyk7cC5hZGRfYXJndW1lbnQoJy0td29yaycscmVxdWlyZWQ9VHJ1ZSk7cC5hZGRfYXJndW1lbnQoJy0tdGFrZS1vdmVyJyxhY3Rpb249J3N0b3JlX3RydWUnKTthPXAucGFyc2VfYXJncygpCiBzaWduYWwuc2lnbmFsKHNpZ25hbC5TSUdJTlQsc3RvcCk7c2lnbmFsLnNpZ25hbChzaWduYWwuU0lHVEVSTSxzdG9wKQogcnVuKGEpCg=='))
(RUNTIME / 'phase2_report.py').write_bytes(base64.b64decode('IiIiQ1BVIGF1ZGl0IG9mIGNvbXBsZXRlZCBQaGFzZSAyIHJ1bnM7IHZhbGlkYXRpb24gc2VsZWN0cyBleHBvcnRzLCBuZXZlciB0ZXN0LiIiIgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IGFyZ3BhcnNlLG9zLHRpbWUKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwaGFzZTJfdHJhaW5pbmdfZGF0YSBhcyBkCmZyb20gcGhhc2UyX2h1YiBpbXBvcnQgSHViLFJFUE8KCmRlZiBtYWluKGFyZ3MpOgogd29yaz1QYXRoKGFyZ3Mud29yayk7d29yay5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSk7aHViPUh1Yih3b3JrKQogcmV2aXNpb249aHViLnJldmlzaW9uKCk7cHJlZml4PWFyZ3MucHJlZml4LnN0cmlwKCcvJykKIGlmIG5vdCBwcmVmaXg6CiAgIyBGaW5kIGEgdW5pcXVlIHByb3RvY29sIHdpdGggY29tcGxldGVkIHRyYWluaW5nOyBhbWJpZ3VpdHkgaXMgbmV2ZXIgZ3Vlc3NlZC4KICBmcm9tIGh1Z2dpbmdmYWNlX2h1Yi5lcnJvcnMgaW1wb3J0IEVudHJ5Tm90Rm91bmRFcnJvcgogIGVudHJpZXM9aHViLnJldHJ5KGxhbWJkYTpsaXN0KGh1Yi5hcGkubGlzdF9yZXBvX3RyZWUoUkVQTywncGhhc2UyL3RyYWluaW5nLXIxJyxyZXBvX3R5cGU9J2RhdGFzZXQnLHJldmlzaW9uPXJldmlzaW9uLHJlY3Vyc2l2ZT1GYWxzZSkpKQogIGNob2ljZXM9W2UucGF0aCBmb3IgZSBpbiBlbnRyaWVzIGlmIHR5cGUoZSkuX19uYW1lX189PSdSZXBvRm9sZGVyJ10KICBpZiBsZW4oY2hvaWNlcykhPTE6cmFpc2UgVmFsdWVFcnJvcignU2V0IFJVTl9QUkVGSVggdG8gdGhlIGludGVuZGVkIHBoYXNlMi90cmFpbmluZy1yMS88cHJvdG9jb2w+IHByaW50ZWQgYnkgdHJhaW5pbmcuIEZvdW5kICcrc3RyKGNob2ljZXMpKQogIHByZWZpeD1jaG9pY2VzWzBdCiBwcm90b2NvbD1wcmVmaXgucnNwbGl0KCcvJywxKVstMV07aHViLnByZWZpeD1wcmVmaXg7aHViLmxlYXNlcz17fQogcm9vdD1kLmxvY2F0ZShhcmdzLnJvb3QpO3BsYW49ZC5vdmVybGF5KHJvb3QpO3Rlc3Q9e3JbJ2ltYWdlX2lkJ106ciBmb3IgciBpbiBwbGFuWydyb3dzJ10gaWYgclsncm9sZSddPT0ndGVzdCd9CiByZXN1bHRzPVtdO3BlbmRpbmc9W107cmVnaXN0cnk9e307Y29uZGl0aW9ucz1hcmdzLmNvbmRpdGlvbnMuc3BsaXQoJywnKQogZm9yIGpvYiBpbiBkLmpvYnMoY29uZGl0aW9ucyk6CiAgYmFzZT1wcmVmaXgrJy9ydW5zLycram9iWydpZCddO21mPWh1Yi5mZXRjaChiYXNlKycvTUFOSUZFU1QuanNvbicscmV2aXNpb24sVHJ1ZSkKICBpZiBub3QgbWYgb3IgZC5yZWFkKG1mKVsnc3RhdHVzJ10hPSdjb21wbGV0ZWQnOnBlbmRpbmcuYXBwZW5kKGpvYlsnaWQnXSk7Y29udGludWUKICBtYW5pZmVzdD1kLnJlYWQobWYpO2Fzc2VydCBtYW5pZmVzdFsncHJvdG9jb2wnXT09cHJvdG9jb2wgYW5kIG1hbmlmZXN0Wydqb2InXT09am9iIGFuZCBtYW5pZmVzdFsnZXBvY2gnXT09NjAKICBhcnRpZmFjdHM9e30KICBmb3IgbmFtZSBpbiAoJ0ZJTkFMLmpzb24nLCdFWFBPUlQuanNvbicpOgogICBwYXRoPWh1Yi5mZXRjaChiYXNlKycvJytuYW1lLHJldmlzaW9uKTthc3NlcnQgZC5zaGEocGF0aCk9PW1hbmlmZXN0WydmaWxlcyddW25hbWVdO2FydGlmYWN0c1tuYW1lXT1kLnJlYWQocGF0aCkKICByZXN1bHQ9YXJ0aWZhY3RzWydGSU5BTC5qc29uJ107ZXhwb3J0PWFydGlmYWN0c1snRVhQT1JULmpzb24nXTtyZWNvcmRzPXJlc3VsdFsndGVzdCddWydyZWNvcmRzJ10KICBleHBlY3RlZD17ayBmb3IgayxyIGluIHRlc3QuaXRlbXMoKSBpZiBqb2JbJ21vZGVsJ10hPSdocm5ldCcgb3IgclsncG9pbnRfa2luZCddPT0naHVtYW4nfQogIGFzc2VydCB7clsnaW1hZ2VfaWQnXSBmb3IgciBpbiByZWNvcmRzfT09ZXhwZWN0ZWQgYW5kIGxlbihyZWNvcmRzKT09bGVuKGV4cGVjdGVkKQogIGFzc2VydCBhbGwoclsndHlyZSddPT10ZXN0W3JbJ2ltYWdlX2lkJ11dWydwaHlzaWNhbF90eXJlX2lkJ10gZm9yIHIgaW4gcmVjb3JkcykKICBpZiBqb2JbJ21vZGVsJ10gaW4gKCdtb2JpbGVuZXR2NCcsJ3Jlc25ldDUwJyk6CiAgIGFzc2VydCBhbGwoclsnbGFiZWwnXT09dGVzdFtyWydpbWFnZV9pZCddXVsnY2xhc3NfaW5kZXgnXSBmb3IgciBpbiByZWNvcmRzKQogICBtZXRyaWM9bGFtYmRhIHJzOmZsb2F0KG5wLm1lYW4oW25wLm1lYW4oW3JbJ3ByZWRpY3Rpb24nXT09YyBmb3IgciBpbiBycyBpZiByWydsYWJlbCddPT1jXSkgZm9yIGMgaW4gc29ydGVkKHtyWydsYWJlbCddIGZvciByIGluIHJzfSldKSkKICBlbGlmIGpvYlsnbW9kZWwnXT09J2hybmV0JzoKICAgYXNzZXJ0IGFsbChucC5hbGxjbG9zZShyWyd0YXJnZXQnXSx0ZXN0W3JbJ2ltYWdlX2lkJ11dWydwb2ludHMnXSkgZm9yIHIgaW4gcmVjb3JkcykKICAgbWV0cmljPWxhbWJkYSByczotZmxvYXQobnAubWVhbihbbnAuYWJzKG5wLmFycmF5KHJbJ3ByZWRpY3Rpb24nXSktclsndGFyZ2V0J10pLm1lYW4oKSBmb3IgciBpbiByc10pKQogIGVsc2U6CiAgIG1ldHJpYz1sYW1iZGEgcnM6ZmxvYXQobnAubWVhbihbeFsnZGljZSddIGZvciByIGluIHJzIGZvciB4IGluIHJbJ3JlZ2lvbnMnXV0pKQogIHNjb3JlPW1ldHJpYyhyZWNvcmRzKTthc3NlcnQgYWJzKHNjb3JlLXJlc3VsdFsndGVzdCddWydzY29yZSddKTwxZS03CiAgcGVyX3R5cmU9e3Q6bWV0cmljKFtyIGZvciByIGluIHJlY29yZHMgaWYgclsndHlyZSddPT10XSkgZm9yIHQgaW4gc29ydGVkKHtyWyd0eXJlJ10gZm9yIHIgaW4gcmVjb3Jkc30pfQogIHJlc3VsdHMuYXBwZW5kKGRpY3Qoam9iPWpvYixzY29yZT1zY29yZSxtZXRyaWM9cmVzdWx0Wyd0ZXN0J11bJ21ldHJpYyddLHBlcl90eXJlPXBlcl90eXJlLHZhbGlkYXRpb25fc2NvcmU9cmVzdWx0WydiZXN0X3Njb3JlJ10pKQogIGtleT1qb2JbJ21vZGVsJ10rJy0nK2pvYlsnY29uZGl0aW9uJ107Y2FuZGlkYXRlPWRpY3Qoam9iPWpvYix2YWxpZGF0aW9uX3Njb3JlPXJlc3VsdFsnYmVzdF9zY29yZSddLHdlaWdodHM9YmFzZSsnL3dlaWdodHMucHQnLHNoYTI1Nj1leHBvcnRbJ3dlaWdodHNfc2hhMjU2J10scmV2aXNpb249cmV2aXNpb24sY29udHJhY3Q9ZXhwb3J0KQogIGlmIGtleSBub3QgaW4gcmVnaXN0cnkgb3IgY2FuZGlkYXRlWyd2YWxpZGF0aW9uX3Njb3JlJ10+cmVnaXN0cnlba2V5XVsndmFsaWRhdGlvbl9zY29yZSddOnJlZ2lzdHJ5W2tleV09Y2FuZGlkYXRlCiBhZ2dyZWdhdGVzPXt9CiBmb3Iga2V5IGluIHJlZ2lzdHJ5OgogIHZhbHVlcz1bclsnc2NvcmUnXSBmb3IgciBpbiByZXN1bHRzIGlmIHJbJ2pvYiddWydtb2RlbCddKyctJytyWydqb2InXVsnY29uZGl0aW9uJ109PWtleV0KICBhZ2dyZWdhdGVzW2tleV09ZGljdChzZWVkcz1sZW4odmFsdWVzKSxtZWFuPWZsb2F0KG5wLm1lYW4odmFsdWVzKSksc3RkPWZsb2F0KG5wLnN0ZCh2YWx1ZXMpKSxjb21wbGV0ZV90aHJlZV9zZWVkcz1sZW4odmFsdWVzKT09MykKIGNvbXBhcmlzb25zPVtdCiBmb3IgbW9kZWwgaW4gZC5NT0RFTFM6CiAgZm9yIHNlZWQgaW4gKDEsMiwzKToKICAgcGFpcj17clsnam9iJ11bJ2NvbmRpdGlvbiddOnIgZm9yIHIgaW4gcmVzdWx0cyBpZiByWydqb2InXVsnbW9kZWwnXT09bW9kZWwgYW5kIHJbJ2pvYiddWydzZWVkJ109PXNlZWR9CiAgIGlmIHNldChwYWlyKT09eydvbGRfb25seScsJ2NvbWJpbmVkJ306Y29tcGFyaXNvbnMuYXBwZW5kKGRpY3QobW9kZWw9bW9kZWwsc2VlZD1zZWVkLGNvbWJpbmVkX21pbnVzX29sZD1wYWlyWydjb21iaW5lZCddWydzY29yZSddLXBhaXJbJ29sZF9vbmx5J11bJ3Njb3JlJ10pKQogcmVwb3J0PWRpY3Qoc3RhdHVzPSdjb21wbGV0ZWQnIGlmIG5vdCBwZW5kaW5nIGVsc2UgJ3BhcnRpYWwnLHJldmlzaW9uPXJldmlzaW9uLHByb3RvY29sPXByb3RvY29sLHJlc3VsdHM9cmVzdWx0cyxhZ2dyZWdhdGVzPWFnZ3JlZ2F0ZXMscGFpcmVkX2NvbXBhcmlzb25zPWNvbXBhcmlzb25zLHBlbmRpbmc9cGVuZGluZyxsaW1pdGF0aW9ucz1wbGFuWydkZWNpc2lvbnMnXSx3b3Jrc3RhdGlvbl9wcm9tb3Rpb249J25vdCBwZXJmb3JtZWQ7IHJlcXVpcmVzIHNlcGFyYXRlIGhhcmR3YXJlL3ZpZGVvIGNoZWNrcycpCiBkLndyaXRlKHdvcmsvJ1JFUE9SVC5qc29uJyxyZXBvcnQpO2Qud3JpdGUod29yay8nRVhQT1JUX1JFR0lTVFJZLmpzb24nLHJlZ2lzdHJ5KQogbWFuaWZlc3Q9e25hbWU6ZC5zaGEod29yay9uYW1lKSBmb3IgbmFtZSBpbiAoJ1JFUE9SVC5qc29uJywnRVhQT1JUX1JFR0lTVFJZLmpzb24nKX07ZC53cml0ZSh3b3JrLydNQU5JRkVTVC5qc29uJyxkaWN0KGZpbGVzPW1hbmlmZXN0LHN0YXR1cz1yZXBvcnRbJ3N0YXR1cyddKSkKIHJ1bj1zdHIoaW50KHRpbWUudGltZSgpKSk7aHViLmNvbW1pdCh7cHJlZml4KycvcmVwb3J0cy8nK3J1bisnLycrbmFtZTp3b3JrL25hbWUgZm9yIG5hbWUgaW4gKCdSRVBPUlQuanNvbicsJ0VYUE9SVF9SRUdJU1RSWS5qc29uJywnTUFOSUZFU1QuanNvbicpfSwnUGhhc2UyIHJlc3VsdCBhdWRpdCcpCiBwcmludCgnUmVwb3J0OicscmVwb3J0WydzdGF0dXMnXSwnY29tcGxldGVkIHJ1bnM6JyxsZW4ocmVzdWx0cyksJ3BlbmRpbmc6JyxsZW4ocGVuZGluZykpCiBwcmludCgnU2F2ZWQgUkVQT1JULmpzb24gYW5kIEVYUE9SVF9SRUdJU1RSWS5qc29uOyBubyBleGlzdGluZyB3b3Jrc3RhdGlvbiBjaGFuZ2VkLicpCmlmIF9fbmFtZV9fPT0nX19tYWluX18nOgogcD1hcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpO3AuYWRkX2FyZ3VtZW50KCctLXdvcmsnLHJlcXVpcmVkPVRydWUpO3AuYWRkX2FyZ3VtZW50KCctLXJvb3QnLGRlZmF1bHQ9JycpO3AuYWRkX2FyZ3VtZW50KCctLXByZWZpeCcsZGVmYXVsdD0nJyk7cC5hZGRfYXJndW1lbnQoJy0tY29uZGl0aW9ucycsZGVmYXVsdD0nY29tYmluZWQnKTttYWluKHAucGFyc2VfYXJncygpKQo='))
(RUNTIME / 'phase2_dataset_verify.py').write_bytes(base64.b64decode('IiIiUG9ydGFibGUgUGhhc2UgMiBpbnRlZ3JpdHkvc3RydWN0dXJlIHZlcmlmaWVyLiBObyB0cmFpbmluZywgZG93bmxvYWRzIG9yIHVwbG9hZHMuIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgY29sbGVjdGlvbnMKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBQSUwgaW1wb3J0IEltYWdlCgoKZGVmIHZlcmlmeShyb290KToKICAgIHJvb3Q9UGF0aChyb290KS5yZXNvbHZlKCkKICAgIGNoZWNrcz17fQogICAgZm9yIGxpbmUgaW4gKHJvb3QvJ1NIQTI1NlNVTVMudHh0JykucmVhZF90ZXh0KGVuY29kaW5nPSd1dGYtOCcpLnNwbGl0bGluZXMoKToKICAgICAgICBkaWdlc3QsbmFtZT1saW5lLnNwbGl0KCcgICcsMSkKICAgICAgICBhc3NlcnQgbmFtZSBub3QgaW4gY2hlY2tzLCdEdXBsaWNhdGUgY2hlY2tzdW0gcGF0aCcKICAgICAgICBjaGVja3NbbmFtZV09ZGlnZXN0CiAgICBmb3IgaSwobmFtZSx3YW50KSBpbiBlbnVtZXJhdGUoY2hlY2tzLml0ZW1zKCkpOgogICAgICAgIHA9KHJvb3QvbmFtZSkucmVzb2x2ZSgpCiAgICAgICAgYXNzZXJ0IHAuaXNfcmVsYXRpdmVfdG8ocm9vdCkgYW5kIHAuaXNfZmlsZSgpLG5hbWUKICAgICAgICBoPWhhc2hsaWIuc2hhMjU2KCkKICAgICAgICB3aXRoIHAub3BlbigncmInKSBhcyBmOgogICAgICAgICAgICBmb3IgYiBpbiBpdGVyKGxhbWJkYTpmLnJlYWQoNCoxMDI0KjEwMjQpLGInJyk6aC51cGRhdGUoYikKICAgICAgICBhc3NlcnQgaC5oZXhkaWdlc3QoKT09d2FudCwgZidDaGVja3N1bSBtaXNtYXRjaDoge25hbWV9JwogICAgICAgIGlmIGkgYW5kIGklNTAwPT0wOnByaW50KGYnSGFzaCBjaGVja2VkIHtpfS97bGVuKGNoZWNrcyl9IGZpbGVzJyxmbHVzaD1UcnVlKQogICAgcmVjb3Jkcz1qc29uLmxvYWRzKChyb290LydtYW5pZmVzdHMvaW1hZ2VzLmpzb24nKS5yZWFkX3RleHQoKSkKICAgIGFzc2VydCBsZW4ocmVjb3Jkcyk9PTU3MCBhbmQgbGVuKHtyWydpbWFnZV9pZCddIGZvciByIGluIHJlY29yZHN9KT09NTcwCiAgICBhc3NlcnQgbGVuKHtyWydpbWFnZV9zaGEyNTYnXSBmb3IgciBpbiByZWNvcmRzfSk9PTU3MAogICAgY291bnRzPWNvbGxlY3Rpb25zLkNvdW50ZXIoclsnZG9tYWluJ10gZm9yIHIgaW4gcmVjb3JkcykKICAgIGFzc2VydCBjb3VudHM9PXsnb3JpZ2luYWxfcGhvdG8nOjQxOCwnbmV3X3ZpZGVvX2ZyYW1lJzoxNTJ9CiAgICBjb25mbGljdF9jb3VudD0wCiAgICBmb3IgciBpbiByZWNvcmRzOgogICAgICAgIGFzc2VydCBjaGVja3NbclsnaW1hZ2VfcGF0aCddXT09clsnaW1hZ2Vfc2hhMjU2J10KICAgICAgICBhc3NlcnQgY2hlY2tzW3JbJ3NvdXJjZV9sYWJlbF9wYXRoJ11dPT1yWydzb3VyY2VfbGFiZWxfc2hhMjU2J10KICAgICAgICB3aXRoIEltYWdlLm9wZW4ocm9vdC9yWydpbWFnZV9wYXRoJ10pIGFzIGltOgogICAgICAgICAgICBpbS5sb2FkKCk7YXNzZXJ0IGltLnNpemU9PShyWyd3aWR0aCddLHJbJ2hlaWdodCddKQogICAgICAgIG1hc2tzPVtdCiAgICAgICAgZm9yIGsgaW4gKCd0eXJlJywndHJlYWQnLCdpZ25vcmUnKToKICAgICAgICAgICAgd2l0aCBJbWFnZS5vcGVuKHJvb3QvcltmJ3trfV9tYXNrJ10pIGFzIGltOgogICAgICAgICAgICAgICAgYT1ucC5hcnJheShpbSk7YXNzZXJ0IGEuc2hhcGU9PShyWydoZWlnaHQnXSxyWyd3aWR0aCddKSBhbmQgc2V0KG5wLnVuaXF1ZShhKSkuaXNzdWJzZXQoezAsMjU1fSkKICAgICAgICAgICAgICAgIG1hc2tzLmFwcGVuZChhPjApCiAgICAgICAgdHlyZSx0cmVhZCxpZ25vcmU9bWFza3MKICAgICAgICBhc3NlcnQgdHlyZS5hbnkoKSBhbmQgdHJlYWQuYW55KCkKICAgICAgICBjb25mbGljdHM9dHJlYWQmfnR5cmUKICAgICAgICBhc3NlcnQgbm90IChjb25mbGljdHMmfmlnbm9yZSkuYW55KCksZidVbm1hc2tlZCBzZW1hbnRpYyBjb25mbGljdDoge3JbImltYWdlX2lkIl19JwogICAgICAgIGFzc2VydCBpbnQoY29uZmxpY3RzLnN1bSgpKT09clsnYW5ub3RhdGlvbl9jb25mbGljdF9waXhlbHMnXQogICAgICAgIGNvbmZsaWN0X2NvdW50Kz1pbnQoY29uZmxpY3RzLnN1bSgpKQogICAgICAgIGFzc2VydCByWydzcGxpdCddPT0nVU5BU1NJR05FRCcsJ1RoaXMgc291cmNlIHJlbGVhc2UgZG9lcyBub3QgY29udGFpbiBhIGZyb3plbiB0cmFpbmluZyBzcGxpdCcKICAgIHBvaW50cz1qc29uLmxvYWRzKChyb290LydnZW9tZXRyeS9vcmlnaW5hbF9odW1hbl9wb2ludHMuanNvbicpLnJlYWRfdGV4dCgpKQogICAgcHJvcG9zYWxzPWpzb24ubG9hZHMoKHJvb3QvJ2dlb21ldHJ5L25ld19wb2ludF9wcm9wb3NhbHMuanNvbicpLnJlYWRfdGV4dCgpKQogICAgYXNzZXJ0IGxlbihwb2ludHMpPT0xMjAgYW5kIGxlbihwcm9wb3NhbHMpPT0xNTIKICAgIGFzc2VydCBhbGwobGVuKHhbJ3BvaW50cyddKT09NiBmb3IgeCBpbiBwb2ludHMrcHJvcG9zYWxzKQogICAgYXNzZXJ0IGFsbChub3QgcFsnaHVtYW5fYWNjZXB0ZWQnXSBmb3Igcm93IGluIHByb3Bvc2FscyBmb3IgcCBpbiByb3dbJ3BvaW50cyddKQogICAgc3RhdHVzPWpzb24ubG9hZHMoKHJvb3QvJ3NwbGl0cy9waGFzZTJfc3BsaXRfc3RhdHVzLmpzb24nKS5yZWFkX3RleHQoKSkKICAgIGFzc2VydCBub3Qgc3RhdHVzWyd0cmFpbmluZ19hbGxvd2VkJ10KICAgIHJlc3VsdD1kaWN0KGludGVncml0eT0nUEFTUycsY2hlY2tzdW1tZWRfZmlsZXM9bGVuKGNoZWNrcyksaW1hZ2VfYW5kX21hc2tfc2V0cz01NzAsCiAgICAgICAgZG9tYWlucz1kaWN0KGNvdW50cyksY29uZmxpY3RfcGl4ZWxzX2V4cGxpY2l0bHlfaWdub3JlZD1jb25mbGljdF9jb3VudCwKICAgICAgICBzb3VyY2VfcGFja2FnZV9yZWFkeT1UcnVlLHRyYWluaW5nX2FsbG93ZWQ9RmFsc2UsCiAgICAgICAgbmV4dF9zdGVwcz1bJ1Jlc29sdmUgZXhhY3Qgb3JpZ2luYWwvdmlkZW8gdHlyZSBpZGVudGl0aWVzIGFuZCBmcmVlemUgc3BsaXRzJywKICAgICAgICAgICAgICAgICAgICAnUmVzb2x2ZSBsYWJlbCBjb25mbGljdHMgZm9yIFlPTE8gb3Igb3RoZXIgdHJhaW5lcnMgd2l0aG91dCBpZ25vcmUgc3VwcG9ydCcsCiAgICAgICAgICAgICAgICAgICAgJ1JldmlldyBkZXJpdmVkIGdlb21ldHJ5IHByb3Bvc2FscycsJ1J1biB0aGUgZm9ydGhjb21pbmcgR1BVL3Jlc3VtZSBwcmVmbGlnaHQnXSkKICAgIHByaW50KGpzb24uZHVtcHMocmVzdWx0LGluZGVudD0yKSxmbHVzaD1UcnVlKQogICAgcmV0dXJuIHJlc3VsdAoKCmlmIF9fbmFtZV9fPT0nX19tYWluX18nOgogICAgcD1hcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpO3AuYWRkX2FyZ3VtZW50KCdyb290JyxuYXJncz0nPycsZGVmYXVsdD1zdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudCkpCiAgICB2ZXJpZnkocC5wYXJzZV9hcmdzKCkucm9vdCkK'))


In [ ]:
# Preserve Kaggle torch, torchvision, NumPy and Pillow. Install only small adapters.
DEPS = WORK / 'deps'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-input',
    '--disable-pip-version-check', '--no-deps', '--upgrade', '--target', str(DEPS),
    'timm==1.0.15', 'ultralytics==8.4.20', 'ultralytics-thop==2.0.18',
    'transformers==4.51.3', 'tokenizers==0.21.4', 'huggingface_hub==0.36.0',
    'safetensors==0.5.3', 'filelock==3.19.1'], timeout=600)

def launch(arguments):
    env = os.environ.copy()
    env['HF_TOKEN'] = TOKEN
    env['PYTHONPATH'] = str(DEPS) + os.pathsep + str(RUNTIME)
    env['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    env['OMP_NUM_THREADS'] = '2'
    env['PYTHONDONTWRITEBYTECODE'] = '1'
    env['YOLO_CONFIG_DIR'] = str(WORK/'yolo_settings')
    (WORK/'yolo_settings'/'Ultralytics').mkdir(parents=True,exist_ok=True)
    child = subprocess.Popen([sys.executable, '-B', '-u', *arguments], env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    del env
    try:
        for line in child.stdout:
            print(line, end='', flush=True)
        code = child.wait()
    except KeyboardInterrupt:
        if child.poll() is None:
            child.send_signal(signal.SIGINT)
            print('Stop requested. Wait for safe checkpoints and the HF upload; do not force-kill the kernel.')
            for line in child.stdout:
                print(line, end='', flush=True)
            child.wait()
        raise
    if code:
        raise RuntimeError('Run stopped with an error. Keep the outputs; local progress and any verified HF snapshots are retained.')


In [ ]:
# Three separate Kaggle sessions: this copy owns only YOLO seed 2.
NUM_WORKERS = 3
WORKER_ID = 1
RUN_CONDITIONS = 'combined'
DATA_ROOT = ''
# ONE-TIME migration after stopping ALL earlier YOLO copies and waiting for uploads.
# Other model notebooks (NB02/NB03/NB05) may keep running.
# Change to False after the first successful start, for ordinary subsequent runs.
TAKE_OVER = True


In [ ]:
arguments = [str(RUNTIME/'phase2_run.py'), 'train', '--work', str(WORK),
    '--root', DATA_ROOT, '--models', 'yolo26m', '--conditions', RUN_CONDITIONS,
    '--worker', str(WORKER_ID), '--workers', str(NUM_WORKERS)]
if TAKE_OVER: arguments.append('--take-over')
launch(arguments)


## Save / stop / resume

Every successful optimizer update gets an atomic local checkpoint containing weights, optimizer, scheduler, AMP scaler, RNG, sampler cursor, history, validation records and best weights. Rerun the same notebook/configuration after a session restart to restore the last hash-verified HF checkpoint. Completed runs are skipped after their result manifests verify.

A single coordinator uploads immutable snapshots roughly every 30 minutes, on run completion and catchable Stop/error. It uses bounded retries, rate budgeting and server backoff. Wait for the printed HF commit after stopping. A forced kernel kill cannot guarantee the final push; progress since the last verified upload can be lost. The watchdog pauses around 8.5 hours.

Checkpoints use measured scratch space; images stay in read-only Kaggle input. The notebook does not assume 1 TB is available. It pauses on insufficient free space instead of deleting unsaved progress. Only one notebook with each worker ID may run at a time for the same jobs. Remote leases fence stale writers. If a crashed session left a lease, confirm the old session is stopped before setting `TAKE_OVER=True` for recovery.

## Label decisions and limitations

SegFormer ignores contradictory pixels. YOLO consumes overlapping instance masks directly, with a documented training-only tyre union with tread; its exclusive-class auxiliary semantic branch is disabled. Source labels are unchanged. HRNet reuses original human points and uses new polygon-derived points as **weak targets weighted 0.25**, never as human-reviewed evaluation truth.

The code was locally tested; the full pretrained T4 smoke results will be produced when you run it. No accuracy improvement, complete three-class test coverage, unseen-video performance or workstation promotion is claimed before those results exist. Keep and share the executed notebook outputs if a check fails.
